# Notebook 13 - New DL architecture experiments

Five independent Optuna studies run in order: FiLM, low-rank fusion, Hadamard/RB-Net, CoGate-Net, and Router + Experts. Each study optimizes mean validation MCC across A and B. Test metrics are recorded for reporting and are not used as the Optuna objective.


In [1]:
from datetime import datetime
from pathlib import Path
import os
import sys
import torch

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'controllers').is_dir() and (p / 'experiments').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate repository root from {cwd}')
sys.path.insert(0, str(REPO_ROOT))

from core.module05v2_fusion_classifier.pipeline import FusionBatchPipeline

BASE_DIR = 'D:/variant_data'
FM_PROFILE_JSON = f'{BASE_DIR}/profiling/fm_profiling_sequence_ablation.json'
FM_SUPPORT_JSON = f'{BASE_DIR}/profiling/fm_sequence_ablation_support.json'
EMBEDDING_DIR = f'{BASE_DIR}/fm_embeddings/sequence_ablation'
GEOMETRY_K = 64
GEOMETRY_DIR = f'{BASE_DIR}/geometry_v2_k{GEOMETRY_K}/{{context}}'
GEOM_PROFILE_JSON = None
GEOM_FEATURE_NAMES = [f'{m}_{f}' for m in ('dna', 'prot') for f in ('LVD_L2', 'LVD_Cosine', 'LID', 'LVD_Relative')]
TARGET_CONFIGS = [
    {'name': 'A_no_geometry', 'ablation': 'bio_core_dna_prot_spliceai', 'active_modalities': ['dna', 'prot', 'bio_core', 'spliceai'], 'geom_feature_names': None},
    {'name': 'B_with_geometry', 'ablation': 'bio_core_dna_prot_spliceai_geom', 'active_modalities': ['dna', 'prot', 'bio_core', 'spliceai', 'geom'], 'geom_feature_names': GEOM_FEATURE_NAMES},
]
DNA_MODEL = {'name': 'nt_v1_500m_dna_1001', 'seq_type': 'dna', 'sequence_length': 1001}
PROTEIN_MODEL = {'name': 'esm2_650m_prot_501', 'seq_type': 'protein', 'sequence_length': 501}
MODELS_SPACE = [DNA_MODEL, PROTEIN_MODEL]
DATASETS = [
    {'name': 'Train3Val_Test', 'train': 'train3', 'val': 'val', 'test': 'test'},
    {'name': 'Train3Val_ClinVarHQ', 'train': 'train3', 'val': 'val', 'test': 'clinvarhq'},
    {'name': 'Train3Val_UniProt', 'train': 'train3', 'val': 'val', 'test': 'uniprot'},
    {'name': 'Train3Val_ProteinGym', 'train': 'train3', 'val': 'val', 'test': 'proteingym'},
]
CONFIG = {'batch_size': 256, 'epochs': 30, 'lr': 1e-4, 'weight_decay': 1e-3, 'lr_factor': 0.5, 'lr_patience': 3, 'early_stop_patience': 6, 'num_workers': 0, 'seed': 42, 'xgb_n_jobs': 1}
EXPERIMENTS = [{'name': 'PyTorch_Concat', 'type': 'pytorch', 'fusion': 'concat'}]
EXPLAINABILITY = {'enable_shap': False, 'enable_lime': False}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

RESUME_EXISTING_RUN = True
RESUME_BATCH_RUN_DIR = 'experiments/module06_architecture_run_20261104_071811'
if RESUME_EXISTING_RUN:
    if RESUME_BATCH_RUN_DIR is None:
        raise ValueError('Set RESUME_BATCH_RUN_DIR when RESUME_EXISTING_RUN=True')
    _resume_path = Path(RESUME_BATCH_RUN_DIR)
    RUN_OUTPUT_DIR = (_resume_path if _resume_path.is_absolute() else REPO_ROOT / _resume_path).resolve()
else:
    RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
    RUN_OUTPUT_DIR = REPO_ROOT / 'experiments' / f'module06_architecture_run_{RUN_STAMP}'
    suffix = 2
    while RUN_OUTPUT_DIR.exists():
        RUN_OUTPUT_DIR = REPO_ROOT / 'experiments' / f'module06_architecture_run_{RUN_STAMP}_{suffix:02d}'
        suffix += 1

pipeline = FusionBatchPipeline(
    base_dir=BASE_DIR, config=CONFIG, datasets=DATASETS, models_space=MODELS_SPACE,
    pooling_strategies=['center'], experiments=EXPERIMENTS, explainability=EXPLAINABILITY,
    fm_profile_json=FM_PROFILE_JSON, geom_profile_json=GEOM_PROFILE_JSON,
    geometry_dir=GEOMETRY_DIR, geom_feature_names=GEOM_FEATURE_NAMES,
    batch_run_dir=str(RUN_OUTPUT_DIR), embedding_dir=EMBEDDING_DIR,
)
print(f'Device: {DEVICE}')
print(f'Batch run directory: {pipeline.batch_run_dir}')
print(f'Target configurations: {[c["name"] for c in TARGET_CONFIGS]}')


c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict_clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Batch run directory: D:\Missense_variant_pathogenicity_prediction\experiments\module06_architecture_run_20261104_071811
Target configurations: ['A_no_geometry', 'B_with_geometry']


## FiLM
Bio/geometry conditions DNA and protein through a FiLM hypernetwork; residual FiLM outputs and the bio embedding feed a residual classifier. Optuna TPE searches the editable ranges in the next code cell. Completed trials, checkpoints, validation/test metrics, and the SQLite study are saved under the FiLM experiment folder.


In [2]:
import importlib
importlib.reload(importlib.import_module("core.module05v2_fusion_classifier.evaluator_profiler"))
importlib.reload(importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner"))
film_experiment_module = importlib.reload(importlib.import_module("core.module06_architecture_experiments.film_experiment"))
FILM_CONFIG = film_experiment_module.get_default_film_config()
FILM_SEARCH_SPACE = film_experiment_module.get_default_film_search_space()
FILM_N_TRIALS = 40
# Edit low/high/step below to tune the ranges before starting this study.
FILM_CONTEXT = {
    'pipeline': pipeline, 'TARGET_CONFIGS': TARGET_CONFIGS, 'datasets': DATASETS,
    'DNA_MODEL': DNA_MODEL, 'PROTEIN_MODEL': PROTEIN_MODEL, 'DEVICE': DEVICE,
}
FILM_RESULTS = film_experiment_module.run_film_optuna_experiment(
    FILM_CONTEXT, base_config=FILM_CONFIG, search_space=FILM_SEARCH_SPACE,
    n_trials=FILM_N_TRIALS, display_fn=display,
)


[I 2026-11-04 07:18:11,949] A new study created in RDB with name: module06_film


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[FiLM] A_no_geometry epoch 1/30 train_loss=0.5219 val_loss=0.3032 val_MCC=0.7095 threshold=0.6351
[FiLM] A_no_geometry epoch 2/30 train_loss=0.3496 val_loss=0.2317 val_MCC=0.7562 threshold=0.6865
[FiLM] A_no_geometry epoch 3/30 train_loss=0.2986 val_loss=0.2126 val_MCC=0.7740 threshold=0.6885
[FiLM] A_no_geometry epoch 4/30 train_loss=0.2751 val_loss=0.1994 val_MCC=0.7880 threshold=0.6990
[FiLM] A_no_geometry epoch 5/30 train_loss=0.2614 val_loss=0.1867 val_MCC=0.7933 threshold=0.7076
[FiLM] A

"[FiLM Optuna] optuna_0000; mean validation MCC=0.8181; A=0.8198; B=0.8164; params={'hidden_dim': 768, 'bio_hidden_dim': 1984, 'head_dim': 768, 'dropout': 0.24147010125782947, 'learning_rate': 3.2735743714072494e-06, 'weight_decay': 6.025215736203862e-06}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[FiLM] A_no_geometry epoch 1/30 train_loss=0.6478 val_loss=0.4912 val_MCC=0.5871 threshold=0.5243
[FiLM] A_no_geometry epoch 2/30 train_loss=0.5436 val_loss=0.4035 val_MCC=0.6491 threshold=0.5765
[FiLM] A_no_geometry epoch 3/30 train_loss=0.4794 val_loss=0.3516 val_MCC=0.6645 threshold=0.6411
[FiLM] A_no_geometry epoch 4/30 train_loss=0.4398 val_loss=0.3183 val_MCC=0.6795 threshold=0.6553
[FiLM] A_no_geometry epoch 5/30 train_loss=0.4119 val_loss=0.2970 val_MCC=0.6895 threshold=0.6934
[FiLM] A

"[FiLM Optuna] optuna_0001; mean validation MCC=0.7994; A=0.7985; B=0.8004; params={'hidden_dim': 128, 'bio_hidden_dim': 1792, 'head_dim': 640, 'dropout': 0.284688668229438, 'learning_rate': 1.1693648443440122e-06, 'weight_decay': 0.07072114131472237}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[FiLM] A_no_geometry epoch 1/30 train_loss=0.4015 val_loss=0.2155 val_MCC=0.7855 threshold=0.7310
[FiLM] A_no_geometry epoch 2/30 train_loss=0.2445 val_loss=0.1834 val_MCC=0.7995 threshold=0.6727
[FiLM] A_no_geometry epoch 3/30 train_loss=0.2221 val_loss=0.1890 val_MCC=0.8046 threshold=0.7197
[FiLM] A_no_geometry epoch 4/30 train_loss=0.2051 val_loss=0.1797 val_MCC=0.8054 threshold=0.6918
[FiLM] A_no_geometry epoch 5/30 train_loss=0.1900 val_loss=0.1742 val_MCC=0.8039 threshold=0.6994
[FiLM] A

... (74 outputs left out)

"[FiLM Optuna] optuna_0039; mean validation MCC=0.8187; A=0.8186; B=0.8188; params={'hidden_dim': 576, 'bio_hidden_dim': 1792, 'head_dim': 832, 'dropout': 0.29094491664094246, 'learning_rate': 6.909461359671151e-06, 'weight_decay': 2.395102829879623e-06}"

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Pooling,DNA_Model,Prot_Model,Network,Architecture,...,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_AUROC,Val_AUPRC,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
0,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,0.8688,0.8386,0.9701,0.8534,0.9753,0.9149,optuna_0000,0,"{""bio_hidden_dim"": 1984, ""dropout"": 0.24147010...",0.81810
60,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,0.8735,0.8321,0.9716,0.8523,0.9761,0.9184,optuna_0030,30,"{""bio_hidden_dim"": 1472, ""dropout"": 0.19580326...",0.81825
78,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,0.8484,0.8585,0.9638,0.8534,0.9761,0.9160,optuna_0039,39,"{""bio_hidden_dim"": 1792, ""dropout"": 0.29094491...",0.81870
58,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,0.8781,0.8267,0.9729,0.8516,0.9757,0.9155,optuna_0029,29,"{""bio_hidden_dim"": 1792, ""dropout"": 0.20887723...",0.81725
54,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,0.8730,0.8309,0.9715,0.8514,0.9767,0.9182,optuna_0027,27,"{""bio_hidden_dim"": 1536, ""dropout"": 0.24567537...",0.81765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,0.8451,0.8432,0.9636,0.8441,0.9730,0.9080,optuna_0006,6,"{""bio_hidden_dim"": 256, ""dropout"": 0.178860235...",0.80720
5,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,0.8617,0.8225,0.9689,0.8417,0.9743,0.9115,optuna_0002,2,"{""bio_hidden_dim"": 448, ""dropout"": 0.077444781...",0.80775
47,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,0.8127,0.8730,0.9526,0.8418,0.9735,0.9091,optuna_0023,23,"{""bio_hidden_dim"": 1728, ""dropout"": 0.35825559...",0.80295
27,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,0.8625,0.8160,0.9693,0.8386,0.9732,0.9072,optuna_0013,13,"{""bio_hidden_dim"": 64, ""dropout"": 0.3647874599...",0.80535


,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Classifier_Peak_VRAM_MB,E2E_GFLOPs/Sample,E2E_Latency_ms/Sample,E2E_Total_Params,E2E_Param_Mem_MB,Max_Peak_VRAM_MB,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
81,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,468.46,819.5741,37.07,1164284769,2273.24,1790.91,optuna_0010,10,"{""bio_hidden_dim"": 1984, ""dropout"": 0.39586659...",0.81305
257,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,211.23,819.5418,37.07,1148162337,2211.73,1790.91,optuna_0032,32,"{""bio_hidden_dim"": 1408, ""dropout"": 0.20333301...",0.81930
97,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,375.17,819.5621,37.07,1158301217,2250.41,1790.91,optuna_0012,12,"{""bio_hidden_dim"": 1856, ""dropout"": 0.26941797...",0.81405
297,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,254.86,819.5474,37.07,1150959201,2222.40,1790.91,optuna_0037,37,"{""bio_hidden_dim"": 1408, ""dropout"": 0.16050401...",0.81140
41,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,272.03,819.5486,37.07,1151536545,2224.60,1790.91,optuna_0005,5,"{""bio_hidden_dim"": 384, ""dropout"": 0.379809787...",0.81635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,327.81,819.5563,38.06,1155372001,2239.24,1790.91,optuna_0025,25,"{""bio_hidden_dim"": 1664, ""dropout"": 0.24374171...",0.81550
190,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,222.18,819.5428,38.05,1148660769,2213.63,1790.91,optuna_0023,23,"{""bio_hidden_dim"": 1728, ""dropout"": 0.35825559...",0.80295
14,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,130.10,819.5321,38.06,1143293985,2193.16,1790.91,optuna_0001,1,"{""bio_hidden_dim"": 1792, ""dropout"": 0.28468866...",0.79945
22,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,433.25,819.5689,38.05,1161699425,2263.37,1790.91,optuna_0002,2,"{""bio_hidden_dim"": 448, ""dropout"": 0.077444781...",0.80775


'FiLM Optuna study complete. SQLite and CSV results: D:\\Missense_variant_pathogenicity_prediction\\experiments\\module06_architecture_run_20261104_071811\\module06_architecture_experiments\\film'

## Low-rank multimodal fusion
The three projected modality factors receive a constant bias feature, are multiplied elementwise, and summed across rank. The classifier consumes this fused vector directly, matching the proposal without an added unimodal skip. Edit the Optuna ranges in the next cell.


In [3]:
import importlib
importlib.reload(importlib.import_module("core.module05v2_fusion_classifier.evaluator_profiler"))
importlib.reload(importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner"))
low_rank_experiment_module = importlib.reload(importlib.import_module("core.module06_architecture_experiments.low_rank_experiment"))
LOW_RANK_CONFIG = low_rank_experiment_module.get_default_low_rank_config()
LOW_RANK_SEARCH_SPACE = low_rank_experiment_module.get_default_low_rank_search_space()
LOW_RANK_N_TRIALS = 50
# Edit low/high/step below to tune the ranges before starting this study.
LOW_RANK_CONTEXT = {
    'pipeline': pipeline, 'TARGET_CONFIGS': TARGET_CONFIGS, 'datasets': DATASETS,
    'DNA_MODEL': DNA_MODEL, 'PROTEIN_MODEL': PROTEIN_MODEL, 'DEVICE': DEVICE,
}
LOW_RANK_RESULTS = low_rank_experiment_module.run_low_rank_optuna_experiment(
    LOW_RANK_CONTEXT, base_config=LOW_RANK_CONFIG,
    search_space=LOW_RANK_SEARCH_SPACE, n_trials=LOW_RANK_N_TRIALS,
    display_fn=display,
)


[I 2026-11-04 07:46:53,561] A new study created in RDB with name: module06_low_rank


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[LowRankFusion] A_no_geometry epoch 1/30 train_loss=0.4898 val_loss=0.3147 val_MCC=0.7348 threshold=0.6323
[LowRankFusion] A_no_geometry epoch 2/30 train_loss=0.3204 val_loss=0.2398 val_MCC=0.7675 threshold=0.6613
[LowRankFusion] A_no_geometry epoch 3/30 train_loss=0.2634 val_loss=0.2156 val_MCC=0.7811 threshold=0.6126
[LowRankFusion] A_no_geometry epoch 4/30 train_loss=0.2313 val_loss=0.2025 val_MCC=0.7910 threshold=0.5829
[LowRankFusion] A_no_geometry epoch 5/30 train_loss=0.2079 val_loss=0.

"[LowRankFusion Optuna] optuna_0000; mean validation MCC=0.8033; A=0.8039; B=0.8026; params={'hidden_dim': 768, 'fusion_dim': 1984, 'rank': 48, 'head_dim': 320, 'dropout': 0.06662736297476243, 'learning_rate': 3.2729742670534234e-06, 'weight_decay': 1.9517224641449486e-06}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[LowRankFusion] A_no_geometry epoch 1/30 train_loss=0.2875 val_loss=0.1942 val_MCC=0.8123 threshold=0.7496
[LowRankFusion] A_no_geometry epoch 2/30 train_loss=0.2143 val_loss=0.1870 val_MCC=0.8023 threshold=0.6714
[LowRankFusion] A_no_geometry epoch 3/30 train_loss=0.1688 val_loss=0.1949 val_MCC=0.7864 threshold=0.8057
[LowRankFusion] A_no_geometry epoch 4/30 train_loss=0.1283 val_loss=0.2452 val_MCC=0.7876 threshold=0.9054
[LowRankFusion] A_no_geometry epoch 5/30 train_loss=0.0942 val_loss=0.

"[LowRankFusion Optuna] optuna_0001; mean validation MCC=0.8168; A=0.8123; B=0.8212; params={'hidden_dim': 1792, 'fusion_dim': 1280, 'rank': 46, 'head_dim': 64, 'dropout': 0.3881143916039878, 'learning_rate': 0.0005596520861285641, 'weight_decay': 1.152644954031562e-05}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[LowRankFusion] A_no_geometry epoch 1/30 train_loss=0.6153 val_loss=0.4591 val_MCC=0.6517 threshold=0.6083
[LowRankFusion] A_no_geometry epoch 2/30 train_loss=0.4586 val_loss=0.3135 val_MCC=0.7215 threshold=0.6272
[LowRankFusion] A_no_geometry epoch 3/30 train_loss=0.3635 val_loss=0.2454 val_MCC=0.7545 threshold=0.6625
[LowRankFusion] A_no_geometry epoch 4/30 train_loss=0.3080 val_loss=0.2145 val_MCC=0.7710 threshold=0.7028
[LowRankFusion] A_no_geometry epoch 5/30 train_loss=0.2761 val_loss=0.

... (94 outputs left out)

"[LowRankFusion Optuna] optuna_0049; mean validation MCC=0.8179; A=0.8176; B=0.8183; params={'hidden_dim': 1728, 'fusion_dim': 1280, 'rank': 32, 'head_dim': 64, 'dropout': 0.3272684901841051, 'learning_rate': 0.0005398250314180868, 'weight_decay': 5.9487843246385304e-05}"

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Pooling,DNA_Model,Prot_Model,Network,Architecture,...,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_AUROC,Val_AUPRC,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
32,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,0.8553,0.8504,0.9661,0.8529,0.9743,0.9156,optuna_0016,16,"{""dropout"": 0.2960730808194991, ""fusion_dim"": ...",0.81810
56,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,0.8599,0.8454,0.9675,0.8526,0.9737,0.9138,optuna_0028,28,"{""dropout"": 0.3780481739419586, ""fusion_dim"": ...",0.81775
98,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,0.8521,0.8527,0.9651,0.8524,0.9746,0.9178,optuna_0049,49,"{""dropout"": 0.3272684901841051, ""fusion_dim"": ...",0.81795
28,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,0.8424,0.8611,0.9620,0.8517,0.9737,0.9137,optuna_0014,14,"{""dropout"": 0.2821256063108284, ""fusion_dim"": ...",0.81675
96,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,0.8506,0.8520,0.9647,0.8513,0.9738,0.9145,optuna_0048,48,"{""dropout"": 0.347396188469338, ""fusion_dim"": 1...",0.81465
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,0.8298,0.8619,0.9583,0.8456,0.9741,0.9137,optuna_0034,34,"{""dropout"": 0.3650302918520911, ""fusion_dim"": ...",0.81020
5,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,0.8325,0.8539,0.9595,0.8431,0.9732,0.9061,optuna_0002,2,"{""dropout"": 0.17561828236363575, ""fusion_dim"":...",0.80290
7,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,0.8576,0.8248,0.9677,0.8409,0.9721,0.9056,optuna_0003,3,"{""dropout"": 0.3151445047502404, ""fusion_dim"": ...",0.80285
1,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,0.8321,0.8493,0.9596,0.8406,0.9727,0.9071,optuna_0000,0,"{""dropout"": 0.06662736297476243, ""fusion_dim"":...",0.80325


,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Classifier_Peak_VRAM_MB,E2E_GFLOPs/Sample,E2E_Latency_ms/Sample,E2E_Total_Params,E2E_Param_Mem_MB,Max_Peak_VRAM_MB,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
217,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,5811.60,820.2375,37.10,1496176673,3539.30,5811.60,optuna_0027,27,"{""dropout"": 0.31502457238713133, ""fusion_dim"":...",0.81490
249,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,4594.26,820.0832,37.08,1418995297,3244.88,4594.26,optuna_0031,31,"{""dropout"": 0.38312241215150866, ""fusion_dim"":...",0.81515
129,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,6647.44,820.3356,37.11,1545258337,3726.53,6647.44,optuna_0016,16,"{""dropout"": 0.2960730808194991, ""fusion_dim"": ...",0.81810
281,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,4355.36,820.0586,37.09,1406645665,3197.77,4355.36,optuna_0035,35,"{""dropout"": 0.3835102576746837, ""fusion_dim"": ...",0.81780
121,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,1505.01,819.7017,37.08,1228122913,2516.76,1790.91,optuna_0015,15,"{""dropout"": 0.3885290150236566, ""fusion_dim"": ...",0.81390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,4155.81,820.0327,38.07,1393687905,3148.34,4155.81,optuna_0037,37,"{""dropout"": 0.3683184499026015, ""fusion_dim"": ...",0.81515
374,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,5810.11,820.2373,38.08,1496076577,3538.92,5810.11,optuna_0046,46,"{""dropout"": 0.3280382434264643, ""fusion_dim"": ...",0.81690
190,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,4452.42,820.0679,38.07,1411331297,3215.64,4452.42,optuna_0023,23,"{""dropout"": 0.39209947455149463, ""fusion_dim"":...",0.81145
278,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,2058.99,819.7708,38.06,1262709153,2648.69,2058.99,optuna_0034,34,"{""dropout"": 0.3650302918520911, ""fusion_dim"": ...",0.81020


'LowRankFusion Optuna study complete. SQLite and CSV results: D:\\Missense_variant_pathogenicity_prediction\\experiments\\module06_architecture_run_20261104_071811\\module06_architecture_experiments\\low_rank'

## Hadamard / Residual Bilinear Fusion (RB-Net)
DNA and protein projections form [DNA, protein, DNA*protein], then a fusion projection. Bio/geometry is input-normalized and projected in its own branch; the residual classifier joins sequence and bio representations. Edit the Optuna ranges in the next cell.


In [4]:
import importlib
importlib.reload(importlib.import_module("core.module05v2_fusion_classifier.evaluator_profiler"))
importlib.reload(importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner"))
hadamard_experiment_module = importlib.reload(importlib.import_module("core.module06_architecture_experiments.hadamard_experiment"))
HADAMARD_CONFIG = hadamard_experiment_module.get_default_hadamard_config()
HADAMARD_SEARCH_SPACE = hadamard_experiment_module.get_default_hadamard_search_space()
HADAMARD_N_TRIALS = 50
# Edit low/high/step below to tune the ranges before starting this study.
HADAMARD_CONTEXT = {
    'pipeline': pipeline, 'TARGET_CONFIGS': TARGET_CONFIGS, 'datasets': DATASETS,
    'DNA_MODEL': DNA_MODEL, 'PROTEIN_MODEL': PROTEIN_MODEL, 'DEVICE': DEVICE,
}
HADAMARD_RESULTS = hadamard_experiment_module.run_hadamard_optuna_experiment(
    HADAMARD_CONTEXT, base_config=HADAMARD_CONFIG,
    search_space=HADAMARD_SEARCH_SPACE, n_trials=HADAMARD_N_TRIALS,
    display_fn=display,
)


[I 2026-11-04 09:42:40,724] A new study created in RDB with name: module06_hadamard


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[HadamardFusion] A_no_geometry epoch 1/30 train_loss=0.5233 val_loss=0.3492 val_MCC=0.7045 threshold=0.6108
[HadamardFusion] A_no_geometry epoch 2/30 train_loss=0.3673 val_loss=0.2660 val_MCC=0.7539 threshold=0.6235
[HadamardFusion] A_no_geometry epoch 3/30 train_loss=0.3095 val_loss=0.2354 val_MCC=0.7738 threshold=0.6490
[HadamardFusion] A_no_geometry epoch 4/30 train_loss=0.2801 val_loss=0.2135 val_MCC=0.7809 threshold=0.6694
[HadamardFusion] A_no_geometry epoch 5/30 train_loss=0.2604 val_lo

"[HadamardFusion Optuna] optuna_0000; mean validation MCC=0.8027; A=0.7999; B=0.8054; params={'hidden_dim': 384, 'fusion_dim': 1024, 'bio_hidden_dim': 1536, 'head_dim': 320, 'dropout': 0.06662736297476243, 'learning_rate': 3.2729742670534234e-06, 'weight_decay': 1.9517224641449486e-06}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[HadamardFusion] A_no_geometry epoch 1/30 train_loss=0.2807 val_loss=0.1817 val_MCC=0.8014 threshold=0.5800
[HadamardFusion] A_no_geometry epoch 2/30 train_loss=0.2207 val_loss=0.2062 val_MCC=0.8035 threshold=0.7875
[HadamardFusion] A_no_geometry epoch 3/30 train_loss=0.1793 val_loss=0.1748 val_MCC=0.8027 threshold=0.6423
[HadamardFusion] A_no_geometry epoch 4/30 train_loss=0.1400 val_loss=0.2438 val_MCC=0.7774 threshold=0.8142
[HadamardFusion] A_no_geometry epoch 5/30 train_loss=0.1019 val_lo

"[HadamardFusion Optuna] optuna_0001; mean validation MCC=0.8052; A=0.8035; B=0.8069; params={'hidden_dim': 896, 'fusion_dim': 640, 'bio_hidden_dim': 1472, 'head_dim': 64, 'dropout': 0.3881143916039878, 'learning_rate': 0.0005596520861285641, 'weight_decay': 1.152644954031562e-05}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[HadamardFusion] A_no_geometry epoch 1/30 train_loss=0.5353 val_loss=0.3447 val_MCC=0.6759 threshold=0.5781
[HadamardFusion] A_no_geometry epoch 2/30 train_loss=0.3831 val_loss=0.2570 val_MCC=0.7354 threshold=0.5698
[HadamardFusion] A_no_geometry epoch 3/30 train_loss=0.3228 val_loss=0.2206 val_MCC=0.7687 threshold=0.5742
[HadamardFusion] A_no_geometry epoch 4/30 train_loss=0.2860 val_loss=0.1976 val_MCC=0.7862 threshold=0.6364
[HadamardFusion] A_no_geometry epoch 5/30 train_loss=0.2661 val_lo

... (94 outputs left out)

"[HadamardFusion Optuna] optuna_0049; mean validation MCC=0.8094; A=0.8097; B=0.8091; params={'hidden_dim': 768, 'fusion_dim': 448, 'bio_hidden_dim': 1984, 'head_dim': 192, 'dropout': 0.033069826297279795, 'learning_rate': 0.00016922945700361034, 'weight_decay': 0.005632645021855918}"

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Pooling,DNA_Model,Prot_Model,Network,Architecture,...,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_AUROC,Val_AUPRC,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
78,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,0.8367,0.8588,0.9605,0.8476,0.9739,0.9135,optuna_0039,39,"{""bio_hidden_dim"": 2048, ""dropout"": 0.05554269...",0.81230
88,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,0.8528,0.8397,0.9658,0.8462,0.9737,0.9121,optuna_0044,44,"{""bio_hidden_dim"": 1792, ""dropout"": 0.05261685...",0.81065
96,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,0.8444,0.8489,0.9631,0.8466,0.9734,0.9127,optuna_0048,48,"{""bio_hidden_dim"": 2048, ""dropout"": 0.02895822...",0.81115
54,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,0.8662,0.8248,0.9700,0.8450,0.9736,0.9116,optuna_0027,27,"{""bio_hidden_dim"": 1344, ""dropout"": 0.03366660...",0.80970
98,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,0.8317,0.8619,0.9589,0.8465,0.9731,0.9121,optuna_0049,49,"{""bio_hidden_dim"": 1984, ""dropout"": 0.03306982...",0.80940
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,0.8480,0.8301,0.9649,0.8390,0.9722,0.9088,optuna_0013,13,"{""bio_hidden_dim"": 256, ""dropout"": 0.095646098...",0.80045
11,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,0.8371,0.8374,0.9616,0.8373,0.9731,0.9060,optuna_0005,5,"{""bio_hidden_dim"": 256, ""dropout"": 0.178860235...",0.79640
43,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,0.8216,0.8542,0.9563,0.8376,0.9723,0.9067,optuna_0021,21,"{""bio_hidden_dim"": 576, ""dropout"": 0.116741109...",0.80115
17,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,0.8554,0.8148,0.9675,0.8346,0.9718,0.9064,optuna_0008,8,"{""bio_hidden_dim"": 256, ""dropout"": 0.107858679...",0.79580


,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Classifier_Peak_VRAM_MB,E2E_GFLOPs/Sample,E2E_Latency_ms/Sample,E2E_Total_Params,E2E_Param_Mem_MB,Max_Peak_VRAM_MB,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
313,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,94.78,819.5274,37.07,1140950665,2184.22,1790.91,optuna_0039,39,"{""bio_hidden_dim"": 2048, ""dropout"": 0.05554269...",0.81230
393,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,94.52,819.5274,37.07,1140932873,2184.15,1790.91,optuna_0049,49,"{""bio_hidden_dim"": 1984, ""dropout"": 0.03306982...",0.80940
289,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,76.79,819.5254,37.08,1139961609,2180.45,1790.91,optuna_0036,36,"{""bio_hidden_dim"": 1472, ""dropout"": 0.07621349...",0.80975
249,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,86.71,819.5265,37.07,1140503241,2182.52,1790.91,optuna_0031,31,"{""bio_hidden_dim"": 1856, ""dropout"": 0.03673234...",0.81090
321,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,77.11,819.5252,37.07,1139822345,2179.92,1790.91,optuna_0040,40,"{""bio_hidden_dim"": 1856, ""dropout"": 0.02451318...",0.80840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,75.61,819.5252,38.05,1139854873,2180.04,1790.91,optuna_0013,13,"{""bio_hidden_dim"": 256, ""dropout"": 0.095646098...",0.80045
126,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,118.74,819.5306,38.05,1142535385,2190.27,1790.91,optuna_0015,15,"{""bio_hidden_dim"": 1024, ""dropout"": 0.12972918...",0.80590
78,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,100.61,819.5284,38.05,1141438169,2186.08,1790.91,optuna_0009,9,"{""bio_hidden_dim"": 1216, ""dropout"": 0.12053201...",0.80910
158,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,100.56,819.5284,38.05,1141439065,2186.09,1790.91,optuna_0019,19,"{""bio_hidden_dim"": 576, ""dropout"": 0.092605680...",0.80530


'HadamardFusion Optuna study complete. SQLite and CSV results: D:\\Missense_variant_pathogenicity_prediction\\experiments\\module06_architecture_run_20261104_071811\\module06_architecture_experiments\\hadamard'

## Independent Co-Gating
Two separately parameterized gates both receive the shared context [DNA; protein]; sigmoid gates modulate their respective vectors by (1 + gate). The gated sequence vector is normalized, then joined with the separately projected bio/geometry vector. Edit the Optuna ranges in the next cell.


In [5]:
import importlib
importlib.reload(importlib.import_module("core.module05v2_fusion_classifier.evaluator_profiler"))
importlib.reload(importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner"))
independent_gating_module = importlib.reload(importlib.import_module("core.module06_architecture_experiments.independent_gating_experiment"))
GATING_CONFIG = independent_gating_module.get_default_gating_config()
GATING_SEARCH_SPACE = independent_gating_module.get_default_gating_search_space()
GATING_N_TRIALS = 50
# Edit low/high/step below to tune the ranges before starting this study.
GATING_CONTEXT = {
    'pipeline': pipeline, 'TARGET_CONFIGS': TARGET_CONFIGS, 'datasets': DATASETS,
    'DNA_MODEL': DNA_MODEL, 'PROTEIN_MODEL': PROTEIN_MODEL, 'DEVICE': DEVICE,
}
GATING_RESULTS = independent_gating_module.run_independent_gating_optuna(
    GATING_CONTEXT, base_config=GATING_CONFIG,
    search_space=GATING_SEARCH_SPACE, n_trials=GATING_N_TRIALS,
    display_fn=display,
)


[I 2026-11-04 10:06:08,941] A new study created in RDB with name: module06_independent_gating


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[IndependentGating] A_no_geometry epoch 1/30 train_loss=0.5945 val_loss=0.4262 val_MCC=0.7053 threshold=0.6193
[IndependentGating] A_no_geometry epoch 2/30 train_loss=0.4201 val_loss=0.2927 val_MCC=0.7548 threshold=0.6216
[IndependentGating] A_no_geometry epoch 3/30 train_loss=0.3447 val_loss=0.2411 val_MCC=0.7734 threshold=0.6875
[IndependentGating] A_no_geometry epoch 4/30 train_loss=0.3085 val_loss=0.2156 val_MCC=0.7855 threshold=0.6341
[IndependentGating] A_no_geometry epoch 5/30 train_los

"[IndependentGating Optuna] optuna_0000; mean validation MCC=0.8093; A=0.8089; B=0.8096; params={'hidden_dim': 768, 'bio_hidden_dim': 1984, 'head_dim': 192, 'dropout': 0.24147010125782947, 'learning_rate': 3.2735743714072494e-06, 'weight_decay': 6.025215736203862e-06}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[IndependentGating] A_no_geometry epoch 1/30 train_loss=0.6777 val_loss=0.6078 val_MCC=0.4813 threshold=0.5503
[IndependentGating] A_no_geometry epoch 2/30 train_loss=0.5917 val_loss=0.5081 val_MCC=0.6093 threshold=0.5658
[IndependentGating] A_no_geometry epoch 3/30 train_loss=0.5310 val_loss=0.4403 val_MCC=0.6425 threshold=0.5896
[IndependentGating] A_no_geometry epoch 4/30 train_loss=0.4891 val_loss=0.3926 val_MCC=0.6578 threshold=0.5609
[IndependentGating] A_no_geometry epoch 5/30 train_los

"[IndependentGating Optuna] optuna_0001; mean validation MCC=0.7870; A=0.7849; B=0.7891; params={'hidden_dim': 128, 'bio_hidden_dim': 1792, 'head_dim': 192, 'dropout': 0.284688668229438, 'learning_rate': 1.1693648443440122e-06, 'weight_decay': 0.07072114131472237}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[IndependentGating] A_no_geometry epoch 1/30 train_loss=0.4737 val_loss=0.3212 val_MCC=0.6711 threshold=0.6865
[IndependentGating] A_no_geometry epoch 2/30 train_loss=0.3216 val_loss=0.2526 val_MCC=0.7332 threshold=0.6127
[IndependentGating] A_no_geometry epoch 3/30 train_loss=0.2668 val_loss=0.2235 val_MCC=0.7678 threshold=0.6332
[IndependentGating] A_no_geometry epoch 4/30 train_loss=0.2294 val_loss=0.2009 val_MCC=0.7826 threshold=0.6162
[IndependentGating] A_no_geometry epoch 5/30 train_los

... (94 outputs left out)

"[IndependentGating Optuna] optuna_0049; mean validation MCC=0.8110; A=0.8091; B=0.8128; params={'hidden_dim': 192, 'bio_hidden_dim': 2048, 'head_dim': 192, 'dropout': 0.32292252137773986, 'learning_rate': 5.069119946017533e-05, 'weight_decay': 0.0014362216763029957}"

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Pooling,DNA_Model,Prot_Model,Network,Architecture,...,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_AUROC,Val_AUPRC,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
14,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,IndependentGating,...,0.8554,0.8416,0.9664,0.8484,0.9736,0.9096,optuna_0007,7,"{""bio_hidden_dim"": 1920, ""dropout"": 0.26669630...",0.81100
34,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,IndependentGating,...,0.8525,0.8443,0.9655,0.8484,0.9749,0.9150,optuna_0017,17,"{""bio_hidden_dim"": 1920, ""dropout"": 0.24448789...",0.81100
92,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,IndependentGating,...,0.8529,0.8432,0.9657,0.8480,0.9743,0.9116,optuna_0046,46,"{""bio_hidden_dim"": 1856, ""dropout"": 0.23010751...",0.81085
24,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,IndependentGating,...,0.8481,0.8481,0.9642,0.8481,0.9747,0.9145,optuna_0012,12,"{""bio_hidden_dim"": 2048, ""dropout"": 0.22759955...",0.81100
28,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,IndependentGating,...,0.8436,0.8523,0.9627,0.8480,0.9745,0.9129,optuna_0014,14,"{""bio_hidden_dim"": 2048, ""dropout"": 0.18431722...",0.80925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,IndependentGating,...,0.8354,0.8409,0.9609,0.8381,0.9713,0.9020,optuna_0019,19,"{""bio_hidden_dim"": 1856, ""dropout"": 0.22230358...",0.80100
13,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,IndependentGating,...,0.8434,0.8282,0.9637,0.8357,0.9717,0.9031,optuna_0006,6,"{""bio_hidden_dim"": 256, ""dropout"": 0.178860235...",0.79585
5,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,IndependentGating,...,0.8332,0.8351,0.9606,0.8342,0.9721,0.9045,optuna_0002,2,"{""bio_hidden_dim"": 448, ""dropout"": 0.077444781...",0.79670
3,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,IndependentGating,...,0.8542,0.8026,0.9677,0.8276,0.9703,0.8986,optuna_0001,1,"{""bio_hidden_dim"": 1792, ""dropout"": 0.28468866...",0.78700


,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Classifier_Peak_VRAM_MB,E2E_GFLOPs/Sample,E2E_Latency_ms/Sample,E2E_Total_Params,E2E_Param_Mem_MB,Max_Peak_VRAM_MB,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
9,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,...,50.29,819.5218,37.07,1138146953,2173.53,1790.91,optuna_0001,1,"{""bio_hidden_dim"": 1792, ""dropout"": 0.28468866...",0.78700
193,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,...,64.73,819.5236,37.08,1139020105,2176.86,1790.91,optuna_0024,24,"{""bio_hidden_dim"": 1920, ""dropout"": 0.12285370...",0.80970
177,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,...,95.06,819.5274,37.07,1140932041,2184.15,1790.91,optuna_0022,22,"{""bio_hidden_dim"": 1728, ""dropout"": 0.20348895...",0.80605
385,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,...,54.08,819.5225,37.08,1138480969,2174.80,1790.91,optuna_0048,48,"{""bio_hidden_dim"": 1856, ""dropout"": 0.36991443...",0.80395
113,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,...,51.50,819.5221,37.08,1138267529,2173.99,1790.91,optuna_0014,14,"{""bio_hidden_dim"": 2048, ""dropout"": 0.18431722...",0.80925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,...,46.27,819.5216,38.05,1138040089,2173.12,1790.91,optuna_0015,15,"{""bio_hidden_dim"": 1856, ""dropout"": 0.29355674...",0.80680
102,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,...,79.01,819.5249,38.06,1139696025,2179.44,1790.91,optuna_0012,12,"{""bio_hidden_dim"": 2048, ""dropout"": 0.22759955...",0.81100
142,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,...,99.07,819.5276,38.05,1141037849,2184.56,1790.91,optuna_0017,17,"{""bio_hidden_dim"": 1920, ""dropout"": 0.24448789...",0.81100
38,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,IndependentGating,...,149.51,819.5338,38.05,1144114073,2196.29,1790.91,optuna_0004,4,"{""bio_hidden_dim"": 1664, ""dropout"": 0.20812260...",0.80445


'IndependentGating Optuna study complete. SQLite and CSV results: D:\\Missense_variant_pathogenicity_prediction\\experiments\\module06_architecture_run_20261104_071811\\module06_architecture_experiments\\independent_gating'

## Router + Experts (proposal 3)
DNA, protein, and bio/geometry experts produce logits. A sample-wise router learns three softmax weights; these alpha values are outputs of the router, not fixed Optuna parameters. The base logit receives beta times a joint correction logit. Optuna searches beta initialization, router temperature, capacity, dropout, learning rate, and weight decay.


In [6]:
import importlib
importlib.reload(importlib.import_module("core.module05v2_fusion_classifier.evaluator_profiler"))
importlib.reload(importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner"))
router_expert_module = importlib.reload(importlib.import_module("core.module06_architecture_experiments.router_expert_experiment"))
ROUTER_EXPERT_CONFIG = router_expert_module.get_default_router_expert_config()
ROUTER_EXPERT_SEARCH_SPACE = router_expert_module.get_default_router_expert_search_space()
ROUTER_EXPERT_N_TRIALS = 60
# Alpha weights remain sample-dependent router outputs; edit search ranges if desired.
ROUTER_EXPERT_CONTEXT = {
    'pipeline': pipeline, 'TARGET_CONFIGS': TARGET_CONFIGS, 'datasets': DATASETS,
    'DNA_MODEL': DNA_MODEL, 'PROTEIN_MODEL': PROTEIN_MODEL, 'DEVICE': DEVICE,
}
ROUTER_EXPERT_RESULTS = router_expert_module.run_router_expert_optuna(
    ROUTER_EXPERT_CONTEXT, base_config=ROUTER_EXPERT_CONFIG,
    search_space=ROUTER_EXPERT_SEARCH_SPACE, n_trials=ROUTER_EXPERT_N_TRIALS,
    display_fn=display,
)


[I 2026-11-04 10:40:12,115] A new study created in RDB with name: module06_router_expert


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RouterExpert] A_no_geometry epoch 1/30 train_loss=0.3565 val_loss=0.1679 val_MCC=0.8110 threshold=0.5652
[RouterExpert] A_no_geometry epoch 2/30 train_loss=0.2216 val_loss=0.1886 val_MCC=0.8066 threshold=0.6326
[RouterExpert] A_no_geometry epoch 3/30 train_loss=0.1882 val_loss=0.2027 val_MCC=0.7919 threshold=0.6873
[RouterExpert] A_no_geometry epoch 4/30 train_loss=0.1614 val_loss=0.1961 val_MCC=0.7970 threshold=0.6740
[RouterExpert] A_no_geometry epoch 5/30 train_loss=0.1199 val_loss=0.2760 

"[RouterExpert Optuna] optuna_0000; mean validation MCC=0.8122; A=0.8110; B=0.8134; params={'embedding_dim': 1600, 'expert_hidden_dim': 1024, 'router_hidden_dim': 576, 'correction_hidden_dim': 640, 'router_temperature': 0.3542363488627512, 'beta_init': 0.05, 'dropout': 0.013130875246841967, 'learning_rate': 0.0015911147987600703, 'weight_decay': 0.014528246637516038}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RouterExpert] A_no_geometry epoch 1/30 train_loss=0.3904 val_loss=0.2097 val_MCC=0.8019 threshold=0.5960
[RouterExpert] A_no_geometry epoch 2/30 train_loss=0.2501 val_loss=0.1854 val_MCC=0.8123 threshold=0.6841
[RouterExpert] A_no_geometry epoch 3/30 train_loss=0.2272 val_loss=0.1899 val_MCC=0.8156 threshold=0.7002
[RouterExpert] A_no_geometry epoch 4/30 train_loss=0.2125 val_loss=0.1745 val_MCC=0.8232 threshold=0.6578
[RouterExpert] A_no_geometry epoch 5/30 train_loss=0.1959 val_loss=0.1709 

"[RouterExpert Optuna] optuna_0001; mean validation MCC=0.8218; A=0.8232; B=0.8204; params={'embedding_dim': 960, 'expert_hidden_dim': 192, 'router_hidden_dim': 144, 'correction_hidden_dim': 320, 'router_temperature': 1.0732750416828638, 'beta_init': 0.05, 'dropout': 0.14971292810100822, 'learning_rate': 3.2025852419994026e-05, 'weight_decay': 0.008431013932082471}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RouterExpert] A_no_geometry epoch 1/30 train_loss=0.6236 val_loss=0.5056 val_MCC=0.6800 threshold=0.5574
[RouterExpert] A_no_geometry epoch 2/30 train_loss=0.4938 val_loss=0.3730 val_MCC=0.7470 threshold=0.5765
[RouterExpert] A_no_geometry epoch 3/30 train_loss=0.4058 val_loss=0.2925 val_MCC=0.7663 threshold=0.5912
[RouterExpert] A_no_geometry epoch 4/30 train_loss=0.3480 val_loss=0.2470 val_MCC=0.7785 threshold=0.5760
[RouterExpert] A_no_geometry epoch 5/30 train_loss=0.3132 val_loss=0.2234 

... (114 outputs left out)

"[RouterExpert Optuna] optuna_0059; mean validation MCC=0.8203; A=0.8184; B=0.8222; params={'embedding_dim': 1344, 'expert_hidden_dim': 384, 'router_hidden_dim': 192, 'correction_hidden_dim': 320, 'router_temperature': 0.7960462996549535, 'beta_init': 0.05, 'dropout': 0.09308937214713263, 'learning_rate': 1.0034878478312833e-05, 'weight_decay': 8.485979125151585e-05}"

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Pooling,DNA_Model,Prot_Model,Network,Architecture,...,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_AUROC,Val_AUPRC,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
62,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,RouterExpert,...,0.8492,0.8684,0.9637,0.8587,0.9759,0.9180,optuna_0031,31,"{""beta_init"": 0.05, ""correction_hidden_dim"": 6...",0.82450
86,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,RouterExpert,...,0.8457,0.8722,0.9625,0.8588,0.9759,0.9183,optuna_0043,43,"{""beta_init"": 0.05, ""correction_hidden_dim"": 6...",0.82170
92,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,RouterExpert,...,0.8677,0.8481,0.9695,0.8578,0.9749,0.9155,optuna_0046,46,"{""beta_init"": 0.05, ""correction_hidden_dim"": 6...",0.82390
76,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,RouterExpert,...,0.8591,0.8558,0.9669,0.8574,0.9767,0.9193,optuna_0038,38,"{""beta_init"": 0.05, ""correction_hidden_dim"": 3...",0.82325
108,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,RouterExpert,...,0.8486,0.8665,0.9636,0.8575,0.9761,0.9175,optuna_0054,54,"{""beta_init"": 0.05, ""correction_hidden_dim"": 7...",0.82375
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,RouterExpert,...,0.8512,0.8466,0.9651,0.8489,0.9742,0.9154,optuna_0000,0,"{""beta_init"": 0.05, ""correction_hidden_dim"": 6...",0.81220
19,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,RouterExpert,...,0.8474,0.8500,0.9639,0.8487,0.9737,0.9105,optuna_0009,9,"{""beta_init"": 0.05, ""correction_hidden_dim"": 8...",0.81355
39,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,RouterExpert,...,0.8504,0.8458,0.9649,0.8481,0.9735,0.9133,optuna_0019,19,"{""beta_init"": 0.05, ""correction_hidden_dim"": 3...",0.81260
57,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,RouterExpert,...,0.8803,0.8129,0.9739,0.8453,0.9735,0.9126,optuna_0028,28,"{""beta_init"": 0.05, ""correction_hidden_dim"": 4...",0.81255


,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Classifier_Peak_VRAM_MB,E2E_GFLOPs/Sample,E2E_Latency_ms/Sample,E2E_Total_Params,E2E_Param_Mem_MB,Max_Peak_VRAM_MB,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
345,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,...,169.48,819.5367,37.08,1145583240,2201.89,1790.91,optuna_0043,43,"{""beta_init"": 0.05, ""correction_hidden_dim"": 6...",0.82170
457,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,...,134.46,819.5325,37.08,1143484968,2193.89,1790.91,optuna_0057,57,"{""beta_init"": 0.05, ""correction_hidden_dim"": 6...",0.81470
249,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,...,154.10,819.5348,37.08,1144617480,2198.21,1790.91,optuna_0031,31,"{""beta_init"": 0.05, ""correction_hidden_dim"": 6...",0.82450
465,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,...,166.99,819.5367,37.08,1145596072,2201.94,1790.91,optuna_0058,58,"{""beta_init"": 0.05, ""correction_hidden_dim"": 7...",0.82220
129,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,...,60.77,819.5237,37.07,1139105608,2177.18,1790.91,optuna_0016,16,"{""beta_init"": 0.05, ""correction_hidden_dim"": 3...",0.81510
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,...,169.32,819.5370,38.05,1145750440,2202.53,1790.91,optuna_0021,21,"{""beta_init"": 0.05, ""correction_hidden_dim"": 5...",0.82105
142,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,...,214.74,819.5425,38.06,1148464776,2212.89,1790.91,optuna_0017,17,"{""beta_init"": 0.05, ""correction_hidden_dim"": 5...",0.81755
366,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,...,120.72,819.5307,38.05,1142610568,2190.55,1790.91,optuna_0045,45,"{""beta_init"": 0.05, ""correction_hidden_dim"": 5...",0.82165
206,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RouterExpert,...,150.92,819.5342,38.06,1144358696,2197.22,1790.91,optuna_0025,25,"{""beta_init"": 0.05, ""correction_hidden_dim"": 1...",0.81975


'RouterExpert Optuna study complete. SQLite and CSV results: D:\\Missense_variant_pathogenicity_prediction\\experiments\\module06_architecture_run_20261104_071811\\module06_architecture_experiments\\router_expert'

In [3]:
# One architecture family, four independent fusion studies run sequentially.
# Each study uses [E_ref ; E_alt] from the existing center-pooled FM files.
import importlib

# Refresh the data loader in an already-open notebook kernel.
importlib.reload(importlib.import_module("core.module05v2_fusion_classifier.dataset"))
pipeline_module = importlib.reload(importlib.import_module("core.module05v2_fusion_classifier.pipeline"))
_module06_ref_alt_batch_dir = pipeline.batch_run_dir
pipeline = pipeline_module.FusionBatchPipeline(
    base_dir=BASE_DIR, config=CONFIG, datasets=DATASETS, models_space=MODELS_SPACE,
    pooling_strategies=['center'], experiments=EXPERIMENTS, explainability=EXPLAINABILITY,
    fm_profile_json=FM_PROFILE_JSON, geom_profile_json=GEOM_PROFILE_JSON,
    geometry_dir=GEOMETRY_DIR, geom_feature_names=GEOM_FEATURE_NAMES,
    batch_run_dir=str(_module06_ref_alt_batch_dir), embedding_dir=EMBEDDING_DIR,
)
importlib.reload(importlib.import_module("core.module05v2_fusion_classifier.evaluator_profiler"))
importlib.reload(importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner"))
ref_alt_fusion_module = importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.ref_alt_fusion_experiment")
)
REF_ALT_FUSION_CONFIG = ref_alt_fusion_module.get_default_ref_alt_config()
# Four independent, editable search spaces: concat, cross_attention, transformer, gating.
REF_ALT_FUSION_SEARCH_SPACES = ref_alt_fusion_module.get_default_ref_alt_search_spaces()
REF_ALT_FUSION_N_TRIALS_PER_FUSION = 40
REF_ALT_FUSION_CONTEXT = {
    'pipeline': pipeline, 'TARGET_CONFIGS': TARGET_CONFIGS, 'datasets': DATASETS,
    'DNA_MODEL': DNA_MODEL, 'PROTEIN_MODEL': PROTEIN_MODEL, 'DEVICE': DEVICE,
}
REF_ALT_FUSION_RESULTS = ref_alt_fusion_module.run_ref_alt_fusion_optuna(
    REF_ALT_FUSION_CONTEXT,
    base_config=REF_ALT_FUSION_CONFIG,
    search_spaces=REF_ALT_FUSION_SEARCH_SPACES,
    n_trials=REF_ALT_FUSION_N_TRIALS_PER_FUSION,
    display_fn=display,
)


[I 2026-11-07 14:26:51,184] A new study created in RDB with name: module06_ref_alt_concat


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RefAltConcat_Concat] A_no_geometry epoch 1/30 train_loss=0.6900 val_loss=0.5547 val_MCC=0.4330 threshold=0.5112
[RefAltConcat_Concat] A_no_geometry epoch 2/30 train_loss=0.6342 val_loss=0.4709 val_MCC=0.5719 threshold=0.4807
[RefAltConcat_Concat] A_no_geometry epoch 3/30 train_loss=0.5888 val_loss=0.4141 val_MCC=0.6166 threshold=0.4729
[RefAltConcat_Concat] A_no_geometry epoch 4/30 train_loss=0.5498 val_loss=0.3731 val_MCC=0.6340 threshold=0.4909
[RefAltConcat_Concat] A_no_geometry epoch 5/30

"[RefAltConcat_Concat Optuna] optuna_0000; mean validation MCC=0.7553; A=0.7526; B=0.7581; params={'fusion_dim': 256, 'bio_upsample_dim': 256, 'dropout': 0.35403628889802274, 'learning_rate': 1.1693648443440122e-06, 'weight_decay': 0.13852278657785003}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RefAltConcat_Concat] A_no_geometry epoch 1/30 train_loss=0.4799 val_loss=0.2847 val_MCC=0.7474 threshold=0.5746
[RefAltConcat_Concat] A_no_geometry epoch 2/30 train_loss=0.3148 val_loss=0.2212 val_MCC=0.7821 threshold=0.5531
[RefAltConcat_Concat] A_no_geometry epoch 3/30 train_loss=0.2700 val_loss=0.2063 val_MCC=0.7938 threshold=0.6918
[RefAltConcat_Concat] A_no_geometry epoch 4/30 train_loss=0.2463 val_loss=0.1967 val_MCC=0.7980 threshold=0.7226
[RefAltConcat_Concat] A_no_geometry epoch 5/30

"[RefAltConcat_Concat Optuna] optuna_0001; mean validation MCC=0.8039; A=0.8063; B=0.8016; params={'fusion_dim': 128, 'bio_upsample_dim': 512, 'dropout': 0.06974693032602092, 'learning_rate': 9.212515846231677e-06, 'weight_decay': 8.7517543796123e-05}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RefAltConcat_Concat] A_no_geometry epoch 1/30 train_loss=0.2820 val_loss=0.1713 val_MCC=0.7945 threshold=0.4889
[RefAltConcat_Concat] A_no_geometry epoch 2/30 train_loss=0.2111 val_loss=0.1829 val_MCC=0.7908 threshold=0.6609
[RefAltConcat_Concat] A_no_geometry epoch 3/30 train_loss=0.1840 val_loss=0.2017 val_MCC=0.7860 threshold=0.7125
[RefAltConcat_Concat] A_no_geometry epoch 4/30 train_loss=0.1614 val_loss=0.2417 val_MCC=0.7742 threshold=0.8130
[RefAltConcat_Concat] A_no_geometry epoch 5/30

... (326 outputs left out)

"[RefAltConcat_Gating Optuna] optuna_0039; mean validation MCC=0.8125; A=0.8105; B=0.8146; params={'fusion_dim': 256, 'bio_upsample_dim': 512, 'dropout': 0.07972983332157713, 'learning_rate': 0.0002648969057027087, 'weight_decay': 0.1790863274860222}"

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Pooling,DNA_Model,Prot_Model,Network,Architecture,...,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_AUROC,Val_AUPRC,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
50,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,RefAltConcat_Gating,...,0.8703,0.8321,0.9708,0.8508,0.9739,0.9162,optuna_0025,25,"{""bio_upsample_dim"": 512, ""dropout"": 0.0248135...",0.81505
68,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,RefAltConcat_Gating,...,0.8558,0.8470,0.9664,0.8514,0.9740,0.9163,optuna_0034,34,"{""bio_upsample_dim"": 512, ""dropout"": 0.0156358...",0.81510
70,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,RefAltConcat_Gating,...,0.8592,0.8428,0.9674,0.8509,0.9744,0.9171,optuna_0035,35,"{""bio_upsample_dim"": 512, ""dropout"": 0.0630364...",0.81600
54,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,RefAltConcat_Gating,...,0.8598,0.8420,0.9676,0.8508,0.9743,0.9170,optuna_0027,27,"{""bio_upsample_dim"": 512, ""dropout"": 0.1131100...",0.81400
20,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,RefAltConcat_Gating,...,0.8617,0.8393,0.9683,0.8504,0.9736,0.9155,optuna_0010,10,"{""bio_upsample_dim"": 512, ""dropout"": 0.0418179...",0.81460
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,RefAltConcat_Gating,...,0.8534,0.8125,0.9671,0.8325,0.9690,0.9019,optuna_0018,18,"{""bio_upsample_dim"": 64, ""dropout"": 0.11316716...",0.79250
9,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,RefAltConcat_Gating,...,0.8501,0.8114,0.9663,0.8303,0.9689,0.8985,optuna_0004,4,"{""bio_upsample_dim"": 256, ""dropout"": 0.0979914...",0.79060
11,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,RefAltConcat_Gating,...,0.8421,0.8183,0.9638,0.8300,0.9688,0.9032,optuna_0005,5,"{""bio_upsample_dim"": 256, ""dropout"": 0.4934434...",0.78900
7,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,RefAltConcat_Gating,...,0.8382,0.8206,0.9627,0.8293,0.9676,0.8979,optuna_0003,3,"{""bio_upsample_dim"": 256, ""dropout"": 0.3312611...",0.78970


,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Classifier_Peak_VRAM_MB,E2E_GFLOPs/Sample,E2E_Latency_ms/Sample,E2E_Total_Params,E2E_Param_Mem_MB,Max_Peak_VRAM_MB,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
281,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,...,40.29,819.5210,37.07,1137723553,2171.91,1790.91,optuna_0035,35,"{""bio_upsample_dim"": 512, ""dropout"": 0.0630364...",0.81600
57,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,...,154.53,819.5350,37.07,1144727585,2198.63,1790.91,optuna_0007,7,"{""bio_upsample_dim"": 512, ""dropout"": 0.2806385...",0.79855
257,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,...,40.29,819.5210,37.07,1137723553,2171.91,1790.91,optuna_0032,32,"{""bio_upsample_dim"": 512, ""dropout"": 0.0511914...",0.81420
233,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,...,40.29,819.5210,37.07,1137723553,2171.91,1790.91,optuna_0029,29,"{""bio_upsample_dim"": 512, ""dropout"": 0.2002338...",0.81205
161,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,...,36.63,819.5206,37.07,1137541217,2171.22,1790.91,optuna_0020,20,"{""bio_upsample_dim"": 64, ""dropout"": 0.14099744...",0.81155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,...,150.97,819.5346,38.06,1144545761,2197.94,1790.91,optuna_0016,16,"{""bio_upsample_dim"": 64, ""dropout"": 0.01256497...",0.81285
158,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,...,79.46,819.5258,38.05,1140150241,2181.17,1790.91,optuna_0019,19,"{""bio_upsample_dim"": 64, ""dropout"": 0.05222624...",0.80565
294,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,...,50.76,819.5224,38.05,1138425377,2174.59,1790.91,optuna_0036,36,"{""bio_upsample_dim"": 256, ""dropout"": 0.0900602...",0.81030
126,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,RefAltConcat_Gating,...,81.55,819.5260,38.05,1140229921,2181.47,1790.91,optuna_0015,15,"{""bio_upsample_dim"": 256, ""dropout"": 0.1778214...",0.80595


'RefAltConcat_Gating Optuna study complete. SQLite and CSV results: D:\\Missense_variant_pathogenicity_prediction\\experiments\\module06_architecture_run_20261104_071811\\module06_architecture_experiments\\ref_alt_gating'

In [3]:
# Shared token-level FM extraction for upcoming token architectures.
# This cell is resumable by shard and writes a persistent path index into this batch run.
import importlib
from pathlib import Path

# Reload Module 2 model loading/extraction dependencies and the Module 6 cache writer.
importlib.reload(importlib.import_module("core.module02_fm_embedding_extraction.models"))
importlib.reload(importlib.import_module("core.module02_fm_embedding_extraction.extractor"))
token_embedding_cache_module = importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.token_embedding_cache")
)

# Shared cache lives with FM artifacts; the batch run stores an index pointing to each model/split folder.
TOKEN_EMBEDDING_CACHE_ROOT = Path(BASE_DIR) / "fm_embeddings" / "token_level_cache" / "module06"
TOKEN_EMBEDDING_SPLITS = [
    {"name": "train3", "input_path": str(Path(BASE_DIR) / "train3_full_seq_final.parquet")},
    {"name": "val", "input_path": str(Path(BASE_DIR) / "val_full_seq_final.parquet")},
    {"name": "test", "input_path": str(Path(BASE_DIR) / "test_full_seq_after_vep_final.parquet")},
    {"name": "clinvarhq", "input_path": str(Path(BASE_DIR) / "clinvarhq_full_seq_after_vep_final.parquet")},
    {"name": "uniprot", "input_path": str(Path(BASE_DIR) / "uniprot_full_seq_after_vep_final.parquet")},
    {"name": "proteingym", "input_path": str(Path(BASE_DIR) / "proteingym_full_seq_after_vep_final.parquet")},
]
TOKEN_EMBEDDING_MODELS = [
    {
        "name": "nt_v1_500m_dna_1001",
        "model_id": "InstaDeepAI/nucleotide-transformer-500m-human-ref",
        "model_type": "mlm", "seq_type": "dna", "sequence_length": 1001,
        "ref_col": "ref_seq_1001", "alt_col": "alt_seq_1001",
    },
    {
        "name": "esm2_650m_prot_501",
        "model_id": "facebook/esm2_t33_650M_UR50D",
        "model_type": "mlm", "seq_type": "protein", "sequence_length": 501,
        "ref_col": "prot_ref_seq_501", "alt_col": "prot_alt_seq_501",
    },
]
# Keep batches conservative because full token hidden states are retained per shard.
TOKEN_EMBEDDING_BATCH_SIZES = {
    "nt_v1_500m_dna_1001": 2,
    "esm2_650m_prot_501": 4,
}
TOKEN_EMBEDDING_CACHE_RUN = token_embedding_cache_module.run_token_embedding_extraction(
    model_configs=TOKEN_EMBEDDING_MODELS,
    split_configs=TOKEN_EMBEDDING_SPLITS,
    cache_root=TOKEN_EMBEDDING_CACHE_ROOT,
    batch_run_dir=pipeline.batch_run_dir,
    device=DEVICE,
    batch_sizes=TOKEN_EMBEDDING_BATCH_SIZES,
)
TOKEN_EMBEDDING_CACHE_INDEX = TOKEN_EMBEDDING_CACHE_RUN["index"]
TOKEN_EMBEDDING_CACHE_INDEX_PATH = TOKEN_EMBEDDING_CACHE_RUN["index_path"]
print(f"Token cache index saved for later architecture cells: {TOKEN_EMBEDDING_CACHE_INDEX_PATH}")


[*] Đang nạp Tokenizer: InstaDeepAI/nucleotide-transformer-500m-human-ref
[*] Đang nạp Mô hình (Trọng số float16): InstaDeepAI/nucleotide-transformer-500m-human-ref


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 396/396 [00:00<00:00, 1270.69it/s]


[!] Da patch compatibility cho remote EsmModel: get_head_mask, get_extended_attention_mask.
[+] Nạp mô hình thành công!

[Token cache estimate] nt_v1_500m_dna_1001/train3: ~20.56 GiB; free space at cache root: 748.02 GiB


Token embeddings: train3:   0%|          | 0/11604 [00:00<?, ?it/s]

[Token Mapping] model=InstaDeepAI/nucleotide-transformer-500m-human-ref | seq_type=dna | strategy=dna_token_diff_fallback


Token embeddings: train3: 100%|██████████| 11604/11604 [20:18<00:00,  9.53it/s]


... (23 outputs left out)

Token embeddings: uniprot: 100%|██████████| 334/334 [00:34<00:00,  9.66it/s]


[Token cache] esm2_650m_prot_501/uniprot: 1333 samples, 334 shards -> D:\variant_data\fm_embeddings\token_level_cache\module06\esm2_650m_prot_501\uniprot
[Token cache estimate] esm2_650m_prot_501/proteingym: ~3.81 GiB; free space at cache root: 601.36 GiB


Token embeddings: proteingym: 100%|██████████| 368/368 [00:37<00:00,  9.78it/s]

[Token cache] esm2_650m_prot_501/proteingym: 1472 samples, 368 shards -> D:\variant_data\fm_embeddings\token_level_cache\module06\esm2_650m_prot_501\proteingym
Token cache index saved for later architecture cells: D:\Missense_variant_pathogenicity_prediction\experiments\module06_architecture_run_20261104_071811\module06_architecture_experiments\token_embedding_cache_index.json


## Transformer c?a s? quanh bi?n th?\nOptuna ch?n b?n k?nh k; m?i modality l?y c?a s? g?m k token m?i b?n v? token bi?n th? ? gi?a (2k+1 token). Ref/alt ???c gh?p theo v? tr? t??ng ??i th?nh m?t token c?p ?? tr?nh l?p hai c?a s? g?n gi?ng nhau.

In [3]:
# k is a centered-window radius: k tokens per side plus the variant token (2k+1 positions).
import importlib
from pathlib import Path

token_cache_module = importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.token_embedding_cache")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner")
)
token_topk_module = importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.token_topk_experiment")
)
if "TOKEN_EMBEDDING_CACHE_INDEX" not in globals():
    TOKEN_EMBEDDING_CACHE_INDEX = token_cache_module.load_token_embedding_cache_index(
        pipeline.batch_run_dir
    )

TOKEN_TOPK_CONFIG = dict(token_topk_module.DEFAULT_TOKEN_TOPK_CONFIG)
TOKEN_TOPK_SEARCH_SPACE = dict(token_topk_module.DEFAULT_TOKEN_TOPK_SEARCH_SPACE)
TOKEN_TOPK_N_TRIALS = 40
# Token tensors are much larger than pooled vectors; lower this if GPU memory is limited.
TOKEN_TOPK_BATCH_SIZE = 8
TOKEN_TOPK_CONTEXT = {
    "pipeline": pipeline,
    "TARGET_CONFIGS": TARGET_CONFIGS,
    "datasets": DATASETS,
    "DNA_MODEL": DNA_MODEL,
    "PROTEIN_MODEL": PROTEIN_MODEL,
    "DEVICE": DEVICE,
    "token_embedding_cache_index": TOKEN_EMBEDDING_CACHE_INDEX,
    "token_batch_size": TOKEN_TOPK_BATCH_SIZE,
}
TOKEN_TOPK_RESULTS = token_topk_module.run_token_topk_optuna(
    context=TOKEN_TOPK_CONTEXT,
    base_config=TOKEN_TOPK_CONFIG,
    search_space=TOKEN_TOPK_SEARCH_SPACE,
    n_trials=TOKEN_TOPK_N_TRIALS,
    display_fn=display,
)

[I 2026-11-09 18:20:23,860] A new study created in RDB with name: module06_transformer_variant_centered_window


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.



D:\Missense_variant_pathogenicity_prediction\core\module06_architecture_experiments\token_topk_experiment.py:267: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=int(config["num_layers"]))


[Transformer_VariantCenteredWindow] A_no_geometry epoch 1/30 train_loss=0.2593 val_loss=0.1833 val_MCC=0.8015 threshold=0.6411
[Transformer_VariantCenteredWindow] A_no_geometry epoch 2/30 train_loss=0.2111 val_loss=0.1926 val_MCC=0.7999 threshold=0.6769
[Transformer_VariantCenteredWindow] A_no_geometry epoch 3/30 train_loss=0.1955 val_loss=0.2176 val_MCC=0.7898 threshold=0.7803
[Transformer_VariantCenteredWindow] A_no_geometry epoch 4/30 train_loss=0.1817 val_loss=0.2027 val_MCC=0.7989 threshold=0.7373
[Transformer_VariantCenteredWindow] A_no_geometry epoch 5/30 train_loss=0.1721 val_loss=0.2601 val_MCC=0.7858 threshold=0.8808
[Transformer_VariantCenteredWindow] A_no_geometry epoch 6/30 train_loss=0.1488 val_loss=0.2327 val_MCC=0.7882 threshold=0.8229
[Transformer_VariantCenteredWindow] A_no_geometry epoch 7/30 train_loss=0.1368 val_loss=0.3103 val_MCC=0.7759 threshold=0.8895
[Transformer_VariantCenteredWindow] Evaluating test set: A_no_geometry / Train3Val_Test
[*] Đang nạp dữ liệu bả

D:\Missense_variant_pathogenicity_prediction\core\module06_architecture_experiments\token_topk_experiment.py:267: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=int(config["num_layers"]))


[Transformer_VariantCenteredWindow] B_with_geometry epoch 1/30 train_loss=0.2575 val_loss=0.1784 val_MCC=0.8019 threshold=0.6168
[Transformer_VariantCenteredWindow] B_with_geometry epoch 2/30 train_loss=0.2087 val_loss=0.1880 val_MCC=0.8010 threshold=0.6636
[Transformer_VariantCenteredWindow] B_with_geometry epoch 3/30 train_loss=0.1934 val_loss=0.2084 val_MCC=0.7937 threshold=0.7603
[Transformer_VariantCenteredWindow] B_with_geometry epoch 4/30 train_loss=0.1794 val_loss=0.1941 val_MCC=0.7987 threshold=0.7125
[Transformer_VariantCenteredWindow] B_with_geometry epoch 5/30 train_loss=0.1697 val_loss=0.2441 val_MCC=0.7889 threshold=0.7798
[Transformer_VariantCenteredWindow] B_with_geometry epoch 6/30 train_loss=0.1449 val_loss=0.2287 val_MCC=0.7944 threshold=0.7470
[Transformer_VariantCenteredWindow] B_with_geometry epoch 7/30 train_loss=0.1322 val_loss=0.2917 val_MCC=0.7806 threshold=0.9170
[Transformer_VariantCenteredWindow] Evaluating test set: B_with_geometry / Train3Val_Test
[*] Đan

... (234 outputs left out)

"[Transformer_VariantCenteredWindow Optuna] optuna_0039; mean validation MCC=0.8124; A=0.8104; B=0.8143; params={'k_radius': 4, 'model_dim': 128, 'num_heads': 8, 'num_layers': 3, 'ffn_multiplier': 2, 'dropout': 0.18741455227118595, 'attention_dropout': 0.19272861375420958, 'bio_dim': 64, 'learning_rate': 0.001541840818353319, 'weight_decay': 0.0003555756291329231}"

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Pooling,DNA_Model,Prot_Model,Network,Architecture,...,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_AUROC,Val_AUPRC,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
52,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,Transformer_VariantCenteredWindow,...,0.8636,0.8309,0.9691,0.8469,0.9728,0.9115,optuna_0026,26,"{""attention_dropout"": 0.2639768407203381, ""bio...",0.80945
72,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,Transformer_VariantCenteredWindow,...,0.8784,0.8148,0.9734,0.8454,0.9736,0.9147,optuna_0036,36,"{""attention_dropout"": 0.3608622191330073, ""bio...",0.81310
64,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,Transformer_VariantCenteredWindow,...,0.8752,0.8179,0.9725,0.8456,0.9725,0.9111,optuna_0032,32,"{""attention_dropout"": 0.24977060569436296, ""bi...",0.81040
44,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,Transformer_VariantCenteredWindow,...,0.8646,0.8279,0.9694,0.8458,0.9726,0.9135,optuna_0022,22,"{""attention_dropout"": 0.3171123686982896, ""bio...",0.80950
78,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,Transformer_VariantCenteredWindow,...,0.8344,0.8600,0.9598,0.8470,0.9734,0.9151,optuna_0039,39,"{""attention_dropout"": 0.19272861375420958, ""bi...",0.81235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,Transformer_VariantCenteredWindow,...,0.8733,0.7911,0.9729,0.8302,0.9675,0.9035,optuna_0010,10,"{""attention_dropout"": 0.2456776766893553, ""bio...",0.79645
13,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,Transformer_VariantCenteredWindow,...,0.8299,0.8210,0.9603,0.8254,0.9668,0.8948,optuna_0006,6,"{""attention_dropout"": 0.2010716092915446, ""bio...",0.78260
17,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,Transformer_VariantCenteredWindow,...,0.8140,0.8386,0.9548,0.8261,0.9678,0.8993,optuna_0008,8,"{""attention_dropout"": 0.3044316186226945, ""bio...",0.78250
9,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,Transformer_VariantCenteredWindow,...,0.8229,0.8267,0.9581,0.8248,0.9671,0.8950,optuna_0004,4,"{""attention_dropout"": 0.20342827646588113, ""bi...",0.78780


,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Classifier_Peak_VRAM_MB,E2E_GFLOPs/Sample,E2E_Latency_ms/Sample,E2E_Total_Params,E2E_Param_Mem_MB,Max_Peak_VRAM_MB,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
209,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,...,64.18,819.5381,38.15,1137864201,2172.45,1790.91,optuna_0026,26,"{""attention_dropout"": 0.2639768407203381, ""bio...",0.80945
25,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,...,319.07,822.0459,37.53,1152539273,2228.43,1790.91,optuna_0003,3,"{""attention_dropout"": 0.24931925073102318, ""bi...",0.80310
177,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,...,64.18,819.5381,38.20,1137864201,2172.45,1790.91,optuna_0022,22,"{""attention_dropout"": 0.3171123686982896, ""bio...",0.80950
217,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,...,64.18,819.5381,38.24,1137864201,2172.45,1790.91,optuna_0027,27,"{""attention_dropout"": 0.22475475664911446, ""bi...",0.80565
313,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,...,64.18,819.5381,38.27,1137864201,2172.45,1790.91,optuna_0039,39,"{""attention_dropout"": 0.19272861375420958, ""bi...",0.81235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,...,62.30,819.5357,39.18,1137732249,2171.95,1790.91,optuna_0030,30,"{""attention_dropout"": 0.24895978973424412, ""bi...",0.80155
102,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,...,66.34,819.5405,39.24,1137997209,2172.96,1790.91,optuna_0012,12,"{""attention_dropout"": 0.22020977384664367, ""bi...",0.80375
30,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,...,316.80,822.0459,38.51,1152539801,2228.43,1790.91,optuna_0003,3,"{""attention_dropout"": 0.24931925073102318, ""bi...",0.80310
6,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_VariantCenteredWindow,...,132.37,819.6806,38.48,1142067801,2188.48,1790.91,optuna_0000,0,"{""attention_dropout"": 0.07336180394137352, ""bi...",0.80170


'Transformer_VariantCenteredWindow Optuna study complete. SQLite and CSV results: D:\\Missense_variant_pathogenicity_prediction\\experiments\\module06_architecture_run_20261104_071811\\module06_architecture_experiments\\transformer_variant_centered_window'

## Transformer ba modality: DNA, protein v? bio/geometry
Cell n?y d?ng token cache ?? tr?ch xu?t: DNA v? protein l? c?c c?a s? quanh bi?n th?; m?i bio/geometry feature ???c bi?u di?n th?nh m?t token c? embedding ??nh danh ri?ng. M?t Transformer duy nh?t h?c t??ng t?c gi?a c? ba modality r?i pooling ?? ph?n lo?i. Optuna t?i ?u mean validation MCC tr?n A/B. M?i seed ch?y th?nh m?t study ri?ng; m?c ??nh ch? c? seed 42. B?n c? th? th?m seed v?o list khi mu?n t?ng ng?n s?ch.

In [6]:
# Three-modality Transformer; k_radius is the number of tokens on each side (window length 2k+1).
import importlib

# Refresh shared runner/token-cache modules, then load this architecture.
importlib.reload(
    importlib.import_module("core.module05v2_fusion_classifier.evaluator_profiler")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.token_topk_experiment")
)
transformer_3modal_module = importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.transformer_3modal_experiment")
)
if "TOKEN_EMBEDDING_CACHE_INDEX" not in globals():
    token_cache_module = importlib.import_module(
        "core.module06_architecture_experiments.token_embedding_cache"
    )
    TOKEN_EMBEDDING_CACHE_INDEX = token_cache_module.load_token_embedding_cache_index(
        pipeline.batch_run_dir
    )

TRANSFORMER_3MODAL_CONFIG = dict(
    transformer_3modal_module.DEFAULT_TRANSFORMER_3MODAL_CONFIG
)
TRANSFORMER_3MODAL_SEARCH_SPACE = dict(
    transformer_3modal_module.DEFAULT_TRANSFORMER_3MODAL_SEARCH_SPACE
)
TRANSFORMER_3MODAL_SEARCH_SPACE["k_radius"] = {
    "type": "categorical", "choices": [4, 8, 16]
}
TRANSFORMER_3MODAL_N_TRIALS = 40
TRANSFORMER_3MODAL_SEEDS = [42]  # Add more seeds here for separate confirmation studies.
TRANSFORMER_3MODAL_BATCH_SIZE = 8
if not TRANSFORMER_3MODAL_SEEDS or len(set(TRANSFORMER_3MODAL_SEEDS)) != len(TRANSFORMER_3MODAL_SEEDS):
    raise ValueError("TRANSFORMER_3MODAL_SEEDS must be a non-empty list of unique integers.")

TRANSFORMER_3MODAL_CONTEXT = {
    "pipeline": pipeline,
    "TARGET_CONFIGS": TARGET_CONFIGS,
    "datasets": DATASETS,
    "DNA_MODEL": DNA_MODEL,
    "PROTEIN_MODEL": PROTEIN_MODEL,
    "DEVICE": DEVICE,
    "token_embedding_cache_index": TOKEN_EMBEDDING_CACHE_INDEX,
    "token_batch_size": TRANSFORMER_3MODAL_BATCH_SIZE,
}
TRANSFORMER_3MODAL_RESULTS = {}
for _transformer_seed in TRANSFORMER_3MODAL_SEEDS:
    _transformer_config = dict(TRANSFORMER_3MODAL_CONFIG)
    _transformer_config["seed"] = int(_transformer_seed)
    _transformer_study_key = f"transformer_3modal_seed_{int(_transformer_seed)}"
    print(f"Starting three-modality Transformer study for seed {_transformer_seed}.")
    TRANSFORMER_3MODAL_RESULTS[int(_transformer_seed)] = (
        transformer_3modal_module.run_transformer_3modal_optuna(
            context=TRANSFORMER_3MODAL_CONTEXT,
            base_config=_transformer_config,
            search_space=TRANSFORMER_3MODAL_SEARCH_SPACE,
            n_trials=TRANSFORMER_3MODAL_N_TRIALS,
            display_fn=display,
            experiment_key=_transformer_study_key,
        )
    )

[I 2026-11-11 21:19:18,018] A new study created in RDB with name: module06_transformer_3modal_seed_42


Starting three-modality Transformer study for seed 42.
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.



D:\Missense_variant_pathogenicity_prediction\core\module06_architecture_experiments\transformer_3modal_experiment.py:99: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=int(config["num_layers"]))


[Transformer_ThreeModalities] A_no_geometry epoch 1/30 train_loss=0.3127 val_loss=0.1859 val_MCC=0.7807 threshold=0.5562
[Transformer_ThreeModalities] A_no_geometry epoch 2/30 train_loss=0.2232 val_loss=0.1850 val_MCC=0.8119 threshold=0.6178
[Transformer_ThreeModalities] A_no_geometry epoch 3/30 train_loss=0.2001 val_loss=0.2124 val_MCC=0.8040 threshold=0.6813
[Transformer_ThreeModalities] A_no_geometry epoch 4/30 train_loss=0.1843 val_loss=0.1706 val_MCC=0.7911 threshold=0.5753
[Transformer_ThreeModalities] A_no_geometry epoch 5/30 train_loss=0.1717 val_loss=0.2264 val_MCC=0.7946 threshold=0.7124
[Transformer_ThreeModalities] A_no_geometry epoch 6/30 train_loss=0.1595 val_loss=0.1983 val_MCC=0.8126 threshold=0.6908
[Transformer_ThreeModalities] A_no_geometry epoch 7/30 train_loss=0.1466 val_loss=0.2922 val_MCC=0.7882 threshold=0.9345
[Transformer_ThreeModalities] A_no_geometry epoch 8/30 train_loss=0.1389 val_loss=0.2682 val_MCC=0.7839 threshold=0.9312
[Transformer_ThreeModalities] A_

D:\Missense_variant_pathogenicity_prediction\core\module06_architecture_experiments\transformer_3modal_experiment.py:99: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=int(config["num_layers"]))


[Transformer_ThreeModalities] B_with_geometry epoch 1/30 train_loss=0.3107 val_loss=0.1884 val_MCC=0.7695 threshold=0.5314
[Transformer_ThreeModalities] B_with_geometry epoch 2/30 train_loss=0.2312 val_loss=0.1835 val_MCC=0.8117 threshold=0.6474
[Transformer_ThreeModalities] B_with_geometry epoch 3/30 train_loss=0.2025 val_loss=0.2138 val_MCC=0.8016 threshold=0.7478
[Transformer_ThreeModalities] B_with_geometry epoch 4/30 train_loss=0.1854 val_loss=0.1744 val_MCC=0.7969 threshold=0.5275
[Transformer_ThreeModalities] B_with_geometry epoch 5/30 train_loss=0.1734 val_loss=0.2358 val_MCC=0.7978 threshold=0.8338
[Transformer_ThreeModalities] B_with_geometry epoch 6/30 train_loss=0.1610 val_loss=0.2193 val_MCC=0.8057 threshold=0.7637
[Transformer_ThreeModalities] B_with_geometry epoch 7/30 train_loss=0.1393 val_loss=0.3007 val_MCC=0.7946 threshold=0.9374
[Transformer_ThreeModalities] B_with_geometry epoch 8/30 train_loss=0.1309 val_loss=0.3040 val_MCC=0.7816 threshold=0.9376
[Transformer_Thr

... (177 outputs left out)

[Transformer_ThreeModalities] A_no_geometry epoch 1/30 train_loss=0.2765 val_loss=0.1844 val_MCC=0.7793 threshold=0.5738
[Transformer_ThreeModalities] A_no_geometry epoch 2/30 train_loss=0.2134 val_loss=0.2073 val_MCC=0.8128 threshold=0.6336
[Transformer_ThreeModalities] A_no_geometry epoch 3/30 train_loss=0.1968 val_loss=0.2064 val_MCC=0.7989 threshold=0.7249
[Transformer_ThreeModalities] A_no_geometry epoch 4/30 train_loss=0.1805 val_loss=0.1754 val_MCC=0.8077 threshold=0.6198
[Transformer_ThreeModalities] A_no_geometry epoch 5/30 train_loss=0.1704 val_loss=0.2288 val_MCC=0.7920 threshold=0.7473
[Transformer_ThreeModalities] A_no_geometry epoch 6/30 train_loss=0.1568 val_loss=0.1974 val_MCC=0.8077 threshold=0.7247
[Transformer_ThreeModalities] A_no_geometry epoch 7/30 train_loss=0.1360 val_loss=0.3093 val_MCC=0.7753 threshold=0.9673
[Transformer_ThreeModalities] A_no_geometry epoch 8/30 train_loss=0.1235 val_loss=0.2762 val_MCC=0.7904 threshold=0.9284
[Transformer_ThreeModalities] Ev

D:\Missense_variant_pathogenicity_prediction\core\module06_architecture_experiments\transformer_3modal_experiment.py:99: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=int(config["num_layers"]))


[Transformer_ThreeModalities] B_with_geometry epoch 1/30 train_loss=0.2709 val_loss=0.1731 val_MCC=0.7961 threshold=0.6351
[Transformer_ThreeModalities] B_with_geometry epoch 2/30 train_loss=0.2114 val_loss=0.1944 val_MCC=0.8099 threshold=0.6173
[Transformer_ThreeModalities] B_with_geometry epoch 3/30 train_loss=0.1959 val_loss=0.2222 val_MCC=0.8051 threshold=0.7792
[Transformer_ThreeModalities] B_with_geometry epoch 4/30 train_loss=0.1813 val_loss=0.1991 val_MCC=0.8039 threshold=0.6978


KeyboardInterrupt: 

## Cross-Attention ba modality — cửa sổ cục bộ quanh vị trí biến thể

Cell này dùng cửa sổ token quanh biến thể, không phải center-only và cũng không phải toàn bộ chuỗi. Optuna chọn k_radius trong {4, 8, 16}; mỗi modality có cửa sổ dài 2k+1 gồm token biến thể và k token mỗi phía. Ba nhánh DNA, protein và bio/geometry trao đổi thông tin qua cross-attention, sau đó pooling riêng rồi nối để phân loại. Study key bắt đầu bằng cross_attention_3modal_seed_; kết quả được lưu dưới tên architecture CrossAttention_ThreeModalities_VariantWindow trong cell tổng hợp.

In [3]:
# Cross-Attention with variant-centered local token windows (k_radius in {4, 8, 16}).
import importlib

importlib.reload(importlib.import_module("core.module05v2_fusion_classifier.evaluator_profiler"))
importlib.reload(importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner"))
importlib.reload(importlib.import_module("core.module06_architecture_experiments.token_topk_experiment"))
importlib.reload(importlib.import_module("core.module06_architecture_experiments.transformer_3modal_experiment"))
cross_attention_3modal_module = importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.cross_attention_3modal_experiment")
)
if "TOKEN_EMBEDDING_CACHE_INDEX" not in globals():
    token_cache_module = importlib.import_module(
        "core.module06_architecture_experiments.token_embedding_cache"
    )
    TOKEN_EMBEDDING_CACHE_INDEX = token_cache_module.load_token_embedding_cache_index(
        pipeline.batch_run_dir
    )

CROSS_ATTENTION_3MODAL_CONFIG = dict(
    cross_attention_3modal_module.DEFAULT_CROSS_ATTENTION_3MODAL_CONFIG
)
CROSS_ATTENTION_3MODAL_SEARCH_SPACE = dict(
    cross_attention_3modal_module.DEFAULT_CROSS_ATTENTION_3MODAL_SEARCH_SPACE
)
# Edit search ranges here before running.
CROSS_ATTENTION_3MODAL_SEARCH_SPACE["k_radius"] = {
    "type": "categorical", "choices": [4, 8, 16]
}
CROSS_ATTENTION_3MODAL_N_TRIALS = 10
CROSS_ATTENTION_3MODAL_SEEDS = [42]  # Add seed values for separate studies.
CROSS_ATTENTION_3MODAL_BATCH_SIZE = 8
if (
    not CROSS_ATTENTION_3MODAL_SEEDS
    or len(set(CROSS_ATTENTION_3MODAL_SEEDS)) != len(CROSS_ATTENTION_3MODAL_SEEDS)
):
    raise ValueError("CROSS_ATTENTION_3MODAL_SEEDS must be a non-empty list of unique integers.")

CROSS_ATTENTION_3MODAL_CONTEXT = {
    "pipeline": pipeline,
    "TARGET_CONFIGS": TARGET_CONFIGS,
    "datasets": DATASETS,
    "DNA_MODEL": DNA_MODEL,
    "PROTEIN_MODEL": PROTEIN_MODEL,
    "DEVICE": DEVICE,
    "token_embedding_cache_index": TOKEN_EMBEDDING_CACHE_INDEX,
    "token_batch_size": CROSS_ATTENTION_3MODAL_BATCH_SIZE,
}
CROSS_ATTENTION_3MODAL_RESULTS = {}
for _cross_attention_seed in CROSS_ATTENTION_3MODAL_SEEDS:
    _cross_attention_config = dict(CROSS_ATTENTION_3MODAL_CONFIG)
    _cross_attention_config["seed"] = int(_cross_attention_seed)
    _cross_attention_study_key = f"cross_attention_3modal_seed_{int(_cross_attention_seed)}"
    print(f"Starting CrossAttention_ThreeModalities_VariantWindow, seed={_cross_attention_seed}, k_radius choices={CROSS_ATTENTION_3MODAL_SEARCH_SPACE['k_radius']['choices']} (2k+1 tokens per sequence modality).")
    CROSS_ATTENTION_3MODAL_RESULTS[int(_cross_attention_seed)] = (
        cross_attention_3modal_module.run_cross_attention_3modal_optuna(
            context=CROSS_ATTENTION_3MODAL_CONTEXT,
            base_config=_cross_attention_config,
            search_space=CROSS_ATTENTION_3MODAL_SEARCH_SPACE,
            n_trials=CROSS_ATTENTION_3MODAL_N_TRIALS,
            display_fn=display,
            experiment_key=_cross_attention_study_key,
        )
    )


[I 2026-11-14 00:21:38,994] Using an existing study with `study_name='module06_cross_attention_3modal_seed_42'` instead of creating a new one.


Starting three-modality Cross-Attention study for seed 42.
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RESUME] Reusing CrossAttention_ThreeModalities checkpoint for A_no_geometry
[CrossAttention_ThreeModalities] Reusing saved test metrics: A_no_geometry / Train3Val_Test
[CrossAttention_ThreeModalities] Reusing saved test metrics: A_no_geometry / Train3Val_ClinVarHQ
[CrossAttention_ThreeModalities] Reusing saved test metrics: A_no_geometry / Train3Val_UniProt
[CrossAttention_ThreeModalities] Reusing saved test metrics: A_no_geometry / T

"[CrossAttention_ThreeModalities Optuna] optuna_0001; mean validation MCC=0.8056; A=0.8070; B=0.8043; params={'k_radius': 8, 'model_dim': 512, 'num_heads': 4, 'num_layers': 3, 'ffn_multiplier': 4, 'pooling': 'mean', 'dropout': 0.2665440364437338, 'attention_dropout': 0.18831220328809845, 'feature_dropout': 0.22701814444901136, 'learning_rate': 1.1693648443440122e-06, 'weight_decay': 0.07072114131472237}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RESUME] Reusing CrossAttention_ThreeModalities checkpoint for A_no_geometry
[CrossAttention_ThreeModalities] Reusing saved test metrics: A_no_geometry / Train3Val_Test
[CrossAttention_ThreeModalities] Reusing saved test metrics: A_no_geometry / Train3Val_ClinVarHQ
[CrossAttention_ThreeModalities] Reusing saved test metrics: A_no_geometry / Train3Val_UniProt
[CrossAttention_ThreeModalities] Reusing saved test metrics: A_no_geometry / Train3Val_ProteinGym
[*] Đang nạp dữ liệu bảng (Backbone). A

"[CrossAttention_ThreeModalities Optuna] optuna_0002; mean validation MCC=0.8056; A=0.8070; B=0.8043; params={'k_radius': 8, 'model_dim': 512, 'num_heads': 4, 'num_layers': 3, 'ffn_multiplier': 4, 'pooling': 'mean', 'dropout': 0.2665440364437338, 'attention_dropout': 0.18831220328809845, 'feature_dropout': 0.22701814444901136, 'learning_rate': 1.1693648443440122e-06, 'weight_decay': 0.07072114131472237}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[CrossAttention_ThreeModalities] A_no_geometry epoch 1/30 train_loss=0.2969 val_loss=0.1648 val_MCC=0.7978 threshold=0.3474
[CrossAttention_ThreeModalities] A_no_geometry epoch 2/30 train_loss=0.2282 val_loss=0.1735 val_MCC=0.8114 threshold=0.6592
[CrossAttention_ThreeModalities] A_no_geometry epoch 3/30 train_loss=0.2116 val_loss=0.1926 val_MCC=0.8121 threshold=0.7587
[CrossAttention_ThreeModalities] A_no_geometry epoch 4/30 train_loss=0.1993 val_loss=0.1588 val_MCC=0.8106 threshold=0.4170
[C

... (9 outputs left out)

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[CrossAttention_ThreeModalities] A_no_geometry epoch 1/30 train_loss=0.2982 val_loss=0.1705 val_MCC=0.7953 threshold=0.5339
[CrossAttention_ThreeModalities] A_no_geometry epoch 2/30 train_loss=0.2249 val_loss=0.1844 val_MCC=0.8120 threshold=0.6235
[CrossAttention_ThreeModalities] A_no_geometry epoch 3/30 train_loss=0.2071 val_loss=0.1970 val_MCC=0.8110 threshold=0.7675
[CrossAttention_ThreeModalities] A_no_geometry epoch 4/30 train_loss=0.1955 val_loss=0.1617 val_MCC=0.8157 threshold=0.5819
[C

"[CrossAttention_ThreeModalities Optuna] optuna_0008; mean validation MCC=0.8158; A=0.8157; B=0.8160; params={'k_radius': 8, 'model_dim': 512, 'num_heads': 8, 'num_layers': 3, 'ffn_multiplier': 8, 'pooling': 'mean', 'dropout': 0.07896726488128243, 'attention_dropout': 0.26166895924516614, 'feature_dropout': 0.20582453170688947, 'learning_rate': 1.2368137521979516e-05, 'weight_decay': 2.078699690689779e-06}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[CrossAttention_ThreeModalities] A_no_geometry epoch 1/30 train_loss=0.2748 val_loss=0.1714 val_MCC=0.7958 threshold=0.5406
[CrossAttention_ThreeModalities] A_no_geometry epoch 2/30 train_loss=0.2196 val_loss=0.1900 val_MCC=0.8087 threshold=0.5644
[CrossAttention_ThreeModalities] A_no_geometry epoch 3/30 train_loss=0.2033 val_loss=0.1971 val_MCC=0.8052 threshold=0.7829
[CrossAttention_ThreeModalities] A_no_geometry epoch 4/30 train_loss=0.1904 val_loss=0.1770 val_MCC=0.8075 threshold=0.5736
[C

KeyboardInterrupt: 

## Router + Experts với token DNA/protein: cửa sổ k=16 và toàn chuỗi

Hai Optuna study dùng cùng ba expert, router, nhánh hiệu chỉnh `beta × delta_logit`, seed và search space. DNA/protein expert nhận cặp ref/alt ở từng token; một study dùng cửa sổ variant-centered với `k_radius=16` (33 vị trí), study còn lại dùng tất cả token hợp lệ. Bio/geometry expert giữ nguyên. Token được attention-pool sau projection; token FM đã mang ngữ cảnh nên bước này tránh self-attention toàn chuỗi có chi phí bộ nhớ bậc hai. Kết quả được lưu riêng theo scope để cell tổng hợp cuối tự phát hiện.

In [2]:
# Router + Experts with token-level sequence experts: centered k=16 versus all valid tokens.
import importlib

importlib.reload(
    importlib.import_module("core.module05v2_fusion_classifier.evaluator_profiler")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.token_topk_experiment")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.router_expert_model")
)
router_expert_token_module = importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.router_expert_token_experiment")
)

if "TOKEN_EMBEDDING_CACHE_INDEX" not in globals():
    token_cache_module = importlib.import_module(
        "core.module06_architecture_experiments.token_embedding_cache"
    )
    TOKEN_EMBEDDING_CACHE_INDEX = token_cache_module.load_token_embedding_cache_index(
        pipeline.batch_run_dir
    )

ROUTER_EXPERT_TOKEN_CONFIG = dict(
    router_expert_token_module.DEFAULT_ROUTER_EXPERT_TOKEN_CONFIG
)
ROUTER_EXPERT_TOKEN_SEARCH_SPACE = dict(
    router_expert_token_module.DEFAULT_ROUTER_EXPERT_TOKEN_SEARCH_SPACE
)
# Edit these Optuna ranges before running.
ROUTER_EXPERT_TOKEN_N_TRIALS = 5
ROUTER_EXPERT_TOKEN_SEEDS = [42]  # Add seed values here for separate confirmation studies.
ROUTER_EXPERT_TOKEN_BATCH_SIZE = 8
if (
    not ROUTER_EXPERT_TOKEN_SEEDS
    or len(set(ROUTER_EXPERT_TOKEN_SEEDS)) != len(ROUTER_EXPERT_TOKEN_SEEDS)
):
    raise ValueError("ROUTER_EXPERT_TOKEN_SEEDS must be a non-empty list of unique integers.")

ROUTER_EXPERT_TOKEN_CONTEXT = {
    "pipeline": pipeline,
    "TARGET_CONFIGS": TARGET_CONFIGS,
    "datasets": DATASETS,
    "DNA_MODEL": DNA_MODEL,
    "PROTEIN_MODEL": PROTEIN_MODEL,
    "DEVICE": DEVICE,
    "token_embedding_cache_index": TOKEN_EMBEDDING_CACHE_INDEX,
    "token_batch_size": ROUTER_EXPERT_TOKEN_BATCH_SIZE,
}
ROUTER_EXPERT_TOKEN_RESULTS = {}
for _router_token_seed in ROUTER_EXPERT_TOKEN_SEEDS:
    for _router_token_scope in router_expert_token_module.TOKEN_SCOPES:
        _router_token_config = dict(ROUTER_EXPERT_TOKEN_CONFIG)
        _router_token_config["seed"] = int(_router_token_seed)
        _router_token_study_key = (
            f"router_expert_{_router_token_scope}_seed_{int(_router_token_seed)}"
        )
        print(
            f"Starting Router + Experts study: scope={_router_token_scope}, "
            f"seed={_router_token_seed}."
        )
        ROUTER_EXPERT_TOKEN_RESULTS[
            (_router_token_scope, int(_router_token_seed))
        ] = router_expert_token_module.run_router_expert_token_optuna(
            context=ROUTER_EXPERT_TOKEN_CONTEXT,
            scope=_router_token_scope,
            base_config=_router_token_config,
            search_space=ROUTER_EXPERT_TOKEN_SEARCH_SPACE,
            n_trials=ROUTER_EXPERT_TOKEN_N_TRIALS,
            display_fn=display,
            experiment_key=_router_token_study_key,
        )


Starting Router + Experts study: scope=centered_16, seed=42.


[I 2026-11-14 13:52:15,272] A new study created in RDB with name: module06_router_expert_centered_16_seed_42


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RouterExpert_TokenInput] A_no_geometry epoch 1/30 train_loss=0.3938 val_loss=0.2105 val_MCC=0.7817 threshold=0.5675
[RouterExpert_TokenInput] A_no_geometry epoch 2/30 train_loss=0.2651 val_loss=0.2487 val_MCC=0.7961 threshold=0.7920
[RouterExpert_TokenInput] A_no_geometry epoch 3/30 train_loss=0.2379 val_loss=0.2402 val_MCC=0.8028 threshold=0.7837
[RouterExpert_TokenInput] A_no_geometry epoch 4/30 train_loss=0.2264 val_loss=0.2018 val_MCC=0.8083 threshold=0.7357
[RouterExpert_TokenInput] A_no

"[RouterExpert_TokenInput Optuna] optuna_0000; mean validation MCC=0.8098; A=0.8109; B=0.8088; params={'token_dim': 512, 'embedding_dim': 512, 'expert_hidden_dim': 256, 'router_hidden_dim': 64, 'correction_hidden_dim': 256, 'router_temperature': 1.2421185223120967, 'beta_init': 0.05, 'dropout': 0.11363873852248522, 'learning_rate': 4.031102009377834e-06, 'weight_decay': 4.10012649801668e-05}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RouterExpert_TokenInput] A_no_geometry epoch 1/30 train_loss=0.4632 val_loss=0.2498 val_MCC=0.7501 threshold=0.6042
[RouterExpert_TokenInput] A_no_geometry epoch 2/30 train_loss=0.3205 val_loss=0.2497 val_MCC=0.7763 threshold=0.7429
[RouterExpert_TokenInput] A_no_geometry epoch 3/30 train_loss=0.2793 val_loss=0.2324 val_MCC=0.7867 threshold=0.7592
[RouterExpert_TokenInput] A_no_geometry epoch 4/30 train_loss=0.2572 val_loss=0.2125 val_MCC=0.7916 threshold=0.6987
[RouterExpert_TokenInput] A_no

"[RouterExpert_TokenInput Optuna] optuna_0001; mean validation MCC=0.8051; A=0.8056; B=0.8045; params={'token_dim': 512, 'embedding_dim': 256, 'expert_hidden_dim': 512, 'router_hidden_dim': 128, 'correction_hidden_dim': 256, 'router_temperature': 1.5918343266467259, 'beta_init': 0.25, 'dropout': 0.10968344329055205, 'learning_rate': 1.639597546292718e-06, 'weight_decay': 0.10716937437552486}"

... (14 outputs left out)

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RouterExpert_TokenInput] A_no_geometry epoch 1/30 train_loss=0.4924 val_loss=0.2516 val_MCC=0.7192 threshold=0.6476
[RouterExpert_TokenInput] A_no_geometry epoch 2/30 train_loss=0.3411 val_loss=0.2876 val_MCC=0.7479 threshold=0.8317
[RouterExpert_TokenInput] A_no_geometry epoch 3/30 train_loss=0.3162 val_loss=0.2753 val_MCC=0.7485 threshold=0.8583
[RouterExpert_TokenInput] A_no_geometry epoch 4/30 train_loss=0.3061 val_loss=0.2539 val_MCC=0.7429 threshold=0.7178
[RouterExpert_TokenInput] A_no

"[RouterExpert_TokenInput Optuna] optuna_0002; mean validation MCC=0.7809; A=0.8094; B=0.7523; params={'token_dim': 512, 'embedding_dim': 256, 'expert_hidden_dim': 256, 'router_hidden_dim': 64, 'correction_hidden_dim': 256, 'router_temperature': 0.16533819011891496, 'beta_init': 0.0, 'dropout': 0.2413485977701479, 'learning_rate': 4.075774074807652e-06, 'weight_decay': 0.13797398094467392}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RouterExpert_TokenInput] A_no_geometry epoch 1/30 train_loss=0.5057 val_loss=0.2629 val_MCC=0.6925 threshold=0.6564
[RouterExpert_TokenInput] A_no_geometry epoch 2/30 train_loss=0.3620 val_loss=0.2459 val_MCC=0.7494 threshold=0.7469
[RouterExpert_TokenInput] A_no_geometry epoch 3/30 train_loss=0.3239 val_loss=0.2316 val_MCC=0.7526 threshold=0.7348
[RouterExpert_TokenInput] A_no_geometry epoch 4/30 train_loss=0.3128 val_loss=0.2223 val_MCC=0.7484 threshold=0.6198
[RouterExpert_TokenInput] A_no

KeyboardInterrupt: 

## Transformer ba modality: center pooling đầu vào so với full tokens

Hai Optuna study so sánh cách chuẩn bị DNA/protein token trước cùng một Transformer ba modality. `center` giữ token tại vị trí biến thể cho mỗi modality (ref/alt ghép thành một token) trước Transformer; `full_tokens` giữ toàn bộ token hợp lệ và căn ref/alt theo vị trí biến thể, không pool chuỗi trước Transformer. Bio/geometry tiếp tục thành feature tokens. Cả hai dùng cùng search space/seed; pooling mean/attention sau Transformer vẫn là bước readout để phân loại. Full-token self-attention tốn bộ nhớ hơn, nên batch size đặt thấp hơn.

In [3]:
# Three-modality Transformer: center-token input versus unpooled full-token input.
import importlib

importlib.reload(
    importlib.import_module("core.module05v2_fusion_classifier.evaluator_profiler")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.token_topk_experiment")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.transformer_3modal_experiment")
)
transformer_3modal_scope_module = importlib.reload(
    importlib.import_module(
        "core.module06_architecture_experiments.transformer_3modal_input_scope_experiment"
    )
)
if "TOKEN_EMBEDDING_CACHE_INDEX" not in globals():
    token_cache_module = importlib.import_module(
        "core.module06_architecture_experiments.token_embedding_cache"
    )
    TOKEN_EMBEDDING_CACHE_INDEX = token_cache_module.load_token_embedding_cache_index(
        pipeline.batch_run_dir
    )

TRANSFORMER_3MODAL_SCOPE_CONFIG = dict(
    transformer_3modal_scope_module.DEFAULT_TRANSFORMER_3MODAL_SCOPE_CONFIG
)
TRANSFORMER_3MODAL_SCOPE_SEARCH_SPACE = dict(
    transformer_3modal_scope_module.DEFAULT_TRANSFORMER_3MODAL_SCOPE_SEARCH_SPACE
)
# Keep the same editable search space for both input scopes.
TRANSFORMER_3MODAL_SCOPE_N_TRIALS = 4
TRANSFORMER_3MODAL_SCOPE_SEEDS = [42]  # Add seeds for additional confirmation studies.
TRANSFORMER_3MODAL_SCOPE_BATCH_SIZE = 4
if (
    not TRANSFORMER_3MODAL_SCOPE_SEEDS
    or len(set(TRANSFORMER_3MODAL_SCOPE_SEEDS))
       != len(TRANSFORMER_3MODAL_SCOPE_SEEDS)
):
    raise ValueError("TRANSFORMER_3MODAL_SCOPE_SEEDS must be a non-empty list of unique integers.")

TRANSFORMER_3MODAL_SCOPE_CONTEXT = {
    "pipeline": pipeline,
    "TARGET_CONFIGS": TARGET_CONFIGS,
    "datasets": DATASETS,
    "DNA_MODEL": DNA_MODEL,
    "PROTEIN_MODEL": PROTEIN_MODEL,
    "DEVICE": DEVICE,
    "token_embedding_cache_index": TOKEN_EMBEDDING_CACHE_INDEX,
    "token_batch_size": TRANSFORMER_3MODAL_SCOPE_BATCH_SIZE,
}
TRANSFORMER_3MODAL_SCOPE_RESULTS = {}
for _scope_seed in TRANSFORMER_3MODAL_SCOPE_SEEDS:
    for _input_scope in transformer_3modal_scope_module.INPUT_SCOPES:
        _scope_config = dict(TRANSFORMER_3MODAL_SCOPE_CONFIG)
        _scope_config["seed"] = int(_scope_seed)
        _scope_study_key = f"transformer_3modal_{_input_scope}_seed_{int(_scope_seed)}"
        print(
            f"Starting three-modality Transformer study: "
            f"input_scope={_input_scope}, seed={_scope_seed}."
        )
        TRANSFORMER_3MODAL_SCOPE_RESULTS[
            (_input_scope, int(_scope_seed))
        ] = transformer_3modal_scope_module.run_transformer_3modal_input_scope_optuna(
            context=TRANSFORMER_3MODAL_SCOPE_CONTEXT,
            input_scope=_input_scope,
            base_config=_scope_config,
            search_space=TRANSFORMER_3MODAL_SCOPE_SEARCH_SPACE,
            n_trials=TRANSFORMER_3MODAL_SCOPE_N_TRIALS,
            display_fn=display,
            experiment_key=_scope_study_key,
        )


[I 2026-11-15 05:34:22,498] Using an existing study with `study_name='module06_transformer_3modal_center_seed_42'` instead of creating a new one.


Starting three-modality Transformer study: input_scope=center, seed=42.
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.



D:\Missense_variant_pathogenicity_prediction\core\module06_architecture_experiments\transformer_3modal_input_scope_experiment.py:140: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


[RESUME] Continuing Transformer_ThreeModalities_InputScope A_no_geometry from epoch 3/30
[Transformer_ThreeModalities_InputScope] A_no_geometry epoch 3/30 train_loss=0.2313 val_loss=0.3352 val_MCC=0.8003 threshold=0.9198
[Transformer_ThreeModalities_InputScope] A_no_geometry epoch 4/30 train_loss=0.2189 val_loss=0.2488 val_MCC=0.8106 threshold=0.8236
[Transformer_ThreeModalities_InputScope] A_no_geometry epoch 5/30 train_loss=0.2095 val_loss=0.3090 val_MCC=0.7970 threshold=0.8593
[Transformer_ThreeModalities_InputScope] A_no_geometry epoch 6/30 train_loss=0.2016 val_loss=0.2780 val_MCC=0.8071 threshold=0.9159
[Transformer_ThreeModalities_InputScope] A_no_geometry epoch 7/30 train_loss=0.1942 val_loss=0.3338 val_MCC=0.8096 threshold=0.9379
[Transformer_ThreeModalities_InputScope] A_no_geometry epoch 8/30 train_loss=0.1890 val_loss=0.3406 val_MCC=0.8046 threshold=0.9478
[Transformer_ThreeModalities_InputScope] A_no_geometry epoch 9/30 train_loss=0.1802 val_loss=0.2576 val_MCC=0.7978 thre

D:\Missense_variant_pathogenicity_prediction\core\module06_architecture_experiments\transformer_3modal_input_scope_experiment.py:140: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


[Transformer_ThreeModalities_InputScope] B_with_geometry epoch 1/30 train_loss=0.3282 val_loss=0.2945 val_MCC=0.8042 threshold=0.8781
[Transformer_ThreeModalities_InputScope] B_with_geometry epoch 2/30 train_loss=0.2445 val_loss=0.2971 val_MCC=0.8067 threshold=0.8844
[Transformer_ThreeModalities_InputScope] B_with_geometry epoch 3/30 train_loss=0.2291 val_loss=0.3974 val_MCC=0.7996 threshold=0.9243
[Transformer_ThreeModalities_InputScope] B_with_geometry epoch 4/30 train_loss=0.2150 val_loss=0.2572 val_MCC=0.8049 threshold=0.8642
[Transformer_ThreeModalities_InputScope] B_with_geometry epoch 5/30 train_loss=0.2077 val_loss=0.3106 val_MCC=0.7976 threshold=0.8843
[Transformer_ThreeModalities_InputScope] B_with_geometry epoch 6/30 train_loss=0.1980 val_loss=0.2818 val_MCC=0.8077 threshold=0.9065
[Transformer_ThreeModalities_InputScope] B_with_geometry epoch 7/30 train_loss=0.1919 val_loss=0.3974 val_MCC=0.8031 threshold=0.9480
[Transformer_ThreeModalities_InputScope] B_with_geometry epoch

... (46 outputs left out)

"[Transformer_ThreeModalities_InputScope Optuna] optuna_0003; mean validation MCC=0.7001; A=0.6717; B=0.7285; params={'model_dim': 128, 'num_heads': 8, 'num_layers': 1, 'ffn_multiplier': 2, 'pooling': 'mean', 'dropout': 0.46484340576040256, 'attention_dropout': 0.0353970008207678, 'feature_dropout': 0.058794858725743554, 'learning_rate': 1.4102517175171362e-06, 'weight_decay': 5.3037841592424196e-05}"

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Pooling,DNA_Model,Prot_Model,Network,Architecture,...,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_AUROC,Val_AUPRC,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
2,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,Transformer_ThreeModalities_InputScope,...,0.8643,0.8332,0.9692,0.8485,0.9714,0.9097,optuna_0001,1,"{""attention_dropout"": 0.0798695128633439, ""dro...",0.80185
0,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,Transformer_ThreeModalities_InputScope,...,0.8606,0.8099,0.9691,0.8345,0.9685,0.9044,optuna_0000,0,"{""attention_dropout"": 0.38796394086479774, ""dr...",0.80425
4,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,Transformer_ThreeModalities_InputScope,...,0.8094,0.7621,0.9577,0.7850,0.9579,0.8598,optuna_0002,2,"{""attention_dropout"": 0.19807076404450807, ""dr...",0.73020
6,module06_architecture_run_20261104_071811,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,Transformer_ThreeModalities_InputScope,...,0.7180,0.7540,0.9302,0.7356,0.9401,0.8060,optuna_0003,3,"{""attention_dropout"": 0.0353970008207678, ""dro...",0.70010
1,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,Transformer_ThreeModalities_InputScope,...,0.8573,0.8363,0.9672,0.8466,0.9723,0.9096,optuna_0000,0,"{""attention_dropout"": 0.38796394086479774, ""dr...",0.80425
3,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,Transformer_ThreeModalities_InputScope,...,0.8328,0.8271,0.9609,0.8299,0.9692,0.9016,optuna_0001,1,"{""attention_dropout"": 0.0798695128633439, ""dro...",0.80185
7,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,Transformer_ThreeModalities_InputScope,...,0.8143,0.7429,0.9600,0.7770,0.9536,0.8450,optuna_0003,3,"{""attention_dropout"": 0.0353970008207678, ""dro...",0.70010
5,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,Transformer_ThreeModalities_InputScope,...,0.7554,0.8011,0.9388,0.7776,0.9526,0.8478,optuna_0002,2,"{""attention_dropout"": 0.19807076404450807, ""dr...",0.73020


,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Classifier_Peak_VRAM_MB,E2E_GFLOPs/Sample,E2E_Latency_ms/Sample,E2E_Total_Params,E2E_Param_Mem_MB,Max_Peak_VRAM_MB,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
9,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,...,181.88,819.5593,37.88,1138692770,2175.61,1790.91,optuna_0001,1,"{""attention_dropout"": 0.0798695128633439, ""dro...",0.80185
1,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,...,193.90,819.6159,37.90,1139480738,2178.62,1790.91,optuna_0000,0,"{""attention_dropout"": 0.38796394086479774, ""dr...",0.80425
17,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,...,219.29,819.5439,38.21,1137826978,2172.31,1790.91,optuna_0002,2,"{""attention_dropout"": 0.19807076404450807, ""dr...",0.73020
25,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,...,213.89,819.5343,37.86,1137562018,2171.30,1790.91,optuna_0003,3,"{""attention_dropout"": 0.0353970008207678, ""dro...",0.70010
3,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ProteinGym,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,...,193.90,819.6159,37.76,1139480738,2178.62,1790.91,optuna_0000,0,"{""attention_dropout"": 0.38796394086479774, ""dr...",0.80425
11,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ProteinGym,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,...,181.88,819.5593,37.78,1138692770,2175.61,1790.91,optuna_0001,1,"{""attention_dropout"": 0.0798695128633439, ""dro...",0.80185
19,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ProteinGym,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,...,219.29,819.5439,38.12,1137826978,2172.31,1790.91,optuna_0002,2,"{""attention_dropout"": 0.19807076404450807, ""dr...",0.73020
27,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ProteinGym,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,...,213.89,819.5343,37.78,1137562018,2171.30,1790.91,optuna_0003,3,"{""attention_dropout"": 0.0353970008207678, ""dro...",0.70010
8,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_Test,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,...,181.88,819.5593,37.65,1138692770,2175.61,1790.91,optuna_0001,1,"{""attention_dropout"": 0.0798695128633439, ""dro...",0.80185
0,module06_architecture_run_20261104_071811,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_Test,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,Transformer_ThreeModalities_InputScope,...,193.90,819.6159,37.68,1139480738,2178.62,1790.91,optuna_0000,0,"{""attention_dropout"": 0.38796394086479774, ""dr...",0.80425


'Transformer_ThreeModalities_InputScope Optuna study complete. SQLite and CSV results: D:\\Missense_variant_pathogenicity_prediction\\experiments\\module06_architecture_run_20261104_071811\\module06_architecture_experiments\\transformer_3modal_full_tokens_seed_42'

## Cross-Attention ba modality — CenterToken so với FullTokens

Cell này chạy hai study riêng, có tên rõ trong kết quả: center chỉ đưa token tại vị trí biến thể của DNA/protein vào cross-attention; full_tokens đưa toàn bộ token hợp lệ vào mà không pool chuỗi trước. Pooling mean/attention sau cross-attention vẫn là bước readout. Cả hai dùng cùng search space/seed; full-token dùng batch size nhỏ hơn.

In [2]:
# Three-modality Cross-Attention: center-token input versus unpooled full-token input.
import importlib

importlib.reload(
    importlib.import_module("core.module05v2_fusion_classifier.evaluator_profiler")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.token_topk_experiment")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.transformer_3modal_experiment")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.transformer_3modal_input_scope_experiment")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.cross_attention_3modal_experiment")
)
importlib.reload(
    importlib.import_module("core.module06_architecture_experiments.cross_attention_3modal_input_scope_experiment")
)
cross_attention_3modal_scope_module = importlib.import_module(
    "core.module06_architecture_experiments.cross_attention_3modal_input_scope_experiment"
)
if "TOKEN_EMBEDDING_CACHE_INDEX" not in globals():
    token_cache_module = importlib.import_module(
        "core.module06_architecture_experiments.token_embedding_cache"
    )
    TOKEN_EMBEDDING_CACHE_INDEX = token_cache_module.load_token_embedding_cache_index(
        pipeline.batch_run_dir
    )

CROSS_ATTENTION_3MODAL_SCOPE_CONFIG = dict(
    cross_attention_3modal_scope_module.DEFAULT_CROSS_ATTENTION_3MODAL_SCOPE_CONFIG
)
CROSS_ATTENTION_3MODAL_SCOPE_SEARCH_SPACE = dict(
    cross_attention_3modal_scope_module.DEFAULT_CROSS_ATTENTION_3MODAL_SCOPE_SEARCH_SPACE
)
# Keep the same editable search space for both input scopes.
CROSS_ATTENTION_3MODAL_SCOPE_N_TRIALS = 4
CROSS_ATTENTION_3MODAL_SCOPE_SEEDS = [42]  # Add seeds for additional confirmation studies.
CROSS_ATTENTION_3MODAL_SCOPE_BATCH_SIZE = 4
if (
    not CROSS_ATTENTION_3MODAL_SCOPE_SEEDS
    or len(set(CROSS_ATTENTION_3MODAL_SCOPE_SEEDS))
       != len(CROSS_ATTENTION_3MODAL_SCOPE_SEEDS)
):
    raise ValueError("CROSS_ATTENTION_3MODAL_SCOPE_SEEDS must be a non-empty list of unique integers.")

CROSS_ATTENTION_3MODAL_SCOPE_CONTEXT = {
    "pipeline": pipeline,
    "TARGET_CONFIGS": TARGET_CONFIGS,
    "datasets": DATASETS,
    "DNA_MODEL": DNA_MODEL,
    "PROTEIN_MODEL": PROTEIN_MODEL,
    "DEVICE": DEVICE,
    "token_embedding_cache_index": TOKEN_EMBEDDING_CACHE_INDEX,
    "token_batch_size": CROSS_ATTENTION_3MODAL_SCOPE_BATCH_SIZE,
}
CROSS_ATTENTION_3MODAL_SCOPE_RESULTS = {}
for _cross_scope_seed in CROSS_ATTENTION_3MODAL_SCOPE_SEEDS:
    for _cross_input_scope in cross_attention_3modal_scope_module.INPUT_SCOPES:
        _cross_scope_config = dict(CROSS_ATTENTION_3MODAL_SCOPE_CONFIG)
        _cross_scope_config["seed"] = int(_cross_scope_seed)
        _cross_scope_study_key = (
            f"cross_attention_3modal_{_cross_input_scope}_seed_{int(_cross_scope_seed)}"
        )
        print(
            f"Starting three-modality Cross-Attention study: "
            f"input_scope={_cross_input_scope}, seed={_cross_scope_seed}."
        )
        CROSS_ATTENTION_3MODAL_SCOPE_RESULTS[
            (_cross_input_scope, int(_cross_scope_seed))
        ] = cross_attention_3modal_scope_module.run_cross_attention_3modal_input_scope_optuna(
            context=CROSS_ATTENTION_3MODAL_SCOPE_CONTEXT,
            input_scope=_cross_input_scope,
            base_config=_cross_scope_config,
            search_space=CROSS_ATTENTION_3MODAL_SCOPE_SEARCH_SPACE,
            n_trials=CROSS_ATTENTION_3MODAL_SCOPE_N_TRIALS,
            display_fn=display,
            experiment_key=_cross_scope_study_key,
        )


Starting three-modality Cross-Attention study: input_scope=center, seed=42.


[I 2026-11-16 11:16:46,815] Using an existing study with `study_name='module06_cross_attention_3modal_center_seed_42'` instead of creating a new one.


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RESUME] Reusing CrossAttention_ThreeModalities_CenterToken checkpoint for A_no_geometry
[CrossAttention_ThreeModalities_CenterToken] Reusing saved test metrics: A_no_geometry / Train3Val_Test
[CrossAttention_ThreeModalities_CenterToken] Reusing saved test metrics: A_no_geometry / Train3Val_ClinVarHQ
[CrossAttention_ThreeModalities_CenterToken] Reusing saved test metrics: A_no_geometry / Train3Val_UniProt
[CrossAttention_ThreeModalities_CenterToken] Reusing saved test metrics: A_no_geometry / 

"[CrossAttention_ThreeModalities_CenterToken Optuna] optuna_0000; mean validation MCC=0.8168; A=0.8189; B=0.8148; params={'model_dim': 256, 'num_heads': 4, 'num_layers': 1, 'ffn_multiplier': 8, 'pooling': 'attention', 'dropout': 0.0592630224331111, 'attention_dropout': 0.38796394086479774, 'feature_dropout': 0.2497327922401265, 'learning_rate': 5.022706182399992e-06, 'weight_decay': 9.201520650562247e-06}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RESUME] Reusing CrossAttention_ThreeModalities_CenterToken checkpoint for A_no_geometry
[CrossAttention_ThreeModalities_CenterToken] Reusing saved test metrics: A_no_geometry / Train3Val_Test
[CrossAttention_ThreeModalities_CenterToken] Reusing saved test metrics: A_no_geometry / Train3Val_ClinVarHQ
[CrossAttention_ThreeModalities_CenterToken] Reusing saved test metrics: A_no_geometry / Train3Val_UniProt
[CrossAttention_ThreeModalities_CenterToken] Reusing saved test metrics: A_no_geometry / 

"[CrossAttention_ThreeModalities_CenterToken Optuna] optuna_0001; mean validation MCC=0.8155; A=0.8107; B=0.8202; params={'model_dim': 256, 'num_heads': 4, 'num_layers': 1, 'ffn_multiplier': 2, 'pooling': 'attention', 'dropout': 0.40332918262685613, 'attention_dropout': 0.0798695128633439, 'feature_dropout': 0.15427033152408348, 'learning_rate': 9.027723761715108e-05, 'weight_decay': 1.7629298492622588e-06}"

... (10 outputs left out)

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RESUME] Reusing CrossAttention_ThreeModalities_FullTokens checkpoint for A_no_geometry
[CrossAttention_ThreeModalities_FullTokens] Reusing saved test metrics: A_no_geometry / Train3Val_Test
[CrossAttention_ThreeModalities_FullTokens] Reusing saved test metrics: A_no_geometry / Train3Val_ClinVarHQ
[CrossAttention_ThreeModalities_FullTokens] Reusing saved test metrics: A_no_geometry / Train3Val_UniProt
[CrossAttention_ThreeModalities_FullTokens] Reusing saved test metrics: A_no_geometry / Train

"[CrossAttention_ThreeModalities_FullTokens Optuna] optuna_0001; mean validation MCC=0.8005; A=0.7998; B=0.8013; params={'model_dim': 256, 'num_heads': 4, 'num_layers': 1, 'ffn_multiplier': 2, 'pooling': 'attention', 'dropout': 0.40332918262685613, 'attention_dropout': 0.0798695128633439, 'feature_dropout': 0.15427033152408348, 'learning_rate': 9.027723761715108e-05, 'weight_decay': 1.7629298492622588e-06}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RESUME] Reusing CrossAttention_ThreeModalities_FullTokens checkpoint for A_no_geometry
[CrossAttention_ThreeModalities_FullTokens] Reusing saved test metrics: A_no_geometry / Train3Val_Test
[CrossAttention_ThreeModalities_FullTokens] Reusing saved test metrics: A_no_geometry / Train3Val_ClinVarHQ
[CrossAttention_ThreeModalities_FullTokens] Reusing saved test metrics: A_no_geometry / Train3Val_UniProt
[CrossAttention_ThreeModalities_FullTokens] Reusing saved test metrics: A_no_geometry / Train

KeyboardInterrupt: 

In [3]:
# Cross-Attention 3 Modality with Skip Connection (Bio-only direct skip).
# DNA, protein, and bio cross-attend; block residuals are disabled in this study.
import importlib

for _cross_skip_module_name in (
    "core.module05v2_fusion_classifier.evaluator_profiler",
    "core.module06_architecture_experiments.architecture_experiment_runner",
    "core.module06_architecture_experiments.token_topk_experiment",
    "core.module06_architecture_experiments.transformer_3modal_experiment",
    "core.module06_architecture_experiments.cross_attention_3modal_experiment",
    "core.module06_architecture_experiments.cross_attention_3modal_bio_skip_experiment",
):
    importlib.reload(importlib.import_module(_cross_skip_module_name))

cross_attention_3modal_bio_skip_module = importlib.import_module(
    "core.module06_architecture_experiments.cross_attention_3modal_bio_skip_experiment"
)
if "TOKEN_EMBEDDING_CACHE_INDEX" not in globals():
    token_cache_module = importlib.import_module(
        "core.module06_architecture_experiments.token_embedding_cache"
    )
    TOKEN_EMBEDDING_CACHE_INDEX = token_cache_module.load_token_embedding_cache_index(
        pipeline.batch_run_dir
    )

CROSS_ATTENTION_3MODAL_BIO_SKIP_CONFIG = dict(
    cross_attention_3modal_bio_skip_module.DEFAULT_CROSS_ATTENTION_3MODAL_BIO_SKIP_CONFIG
)
CROSS_ATTENTION_3MODAL_BIO_SKIP_SEARCH_SPACE = dict(
    cross_attention_3modal_bio_skip_module.DEFAULT_CROSS_ATTENTION_3MODAL_BIO_SKIP_SEARCH_SPACE
)
# Edit the search ranges here before running this study.
CROSS_ATTENTION_3MODAL_BIO_SKIP_N_TRIALS = 3
CROSS_ATTENTION_3MODAL_BIO_SKIP_SEEDS = [42]
CROSS_ATTENTION_3MODAL_BIO_SKIP_BATCH_SIZE = 8
CROSS_ATTENTION_3MODAL_BIO_SKIP_RUN_CONFIG_A = False  # Set True to include A_no_geometry.
CROSS_ATTENTION_3MODAL_BIO_SKIP_TARGET_CONFIGS = [
    config for config in TARGET_CONFIGS
    if CROSS_ATTENTION_3MODAL_BIO_SKIP_RUN_CONFIG_A
    or config.get("name") != "A_no_geometry"
]
if not CROSS_ATTENTION_3MODAL_BIO_SKIP_TARGET_CONFIGS:
    raise ValueError("No target configurations selected for the bio-skip study.")
print("Cross-Attention bio-skip target configs:", [c["name"] for c in CROSS_ATTENTION_3MODAL_BIO_SKIP_TARGET_CONFIGS])
_cross_skip_config_names = {c["name"] for c in CROSS_ATTENTION_3MODAL_BIO_SKIP_TARGET_CONFIGS}
if _cross_skip_config_names == {"A_no_geometry", "B_with_geometry"}:
    CROSS_ATTENTION_3MODAL_BIO_SKIP_OBJECTIVE_NAME = "mean_validation_mcc_across_A_and_B"
elif len(_cross_skip_config_names) == 1:
    CROSS_ATTENTION_3MODAL_BIO_SKIP_OBJECTIVE_NAME = f"validation_mcc_on_{next(iter(_cross_skip_config_names))}"
else:
    CROSS_ATTENTION_3MODAL_BIO_SKIP_OBJECTIVE_NAME = "mean_validation_mcc_over_selected_configs"
if (
    not CROSS_ATTENTION_3MODAL_BIO_SKIP_SEEDS
    or len(set(CROSS_ATTENTION_3MODAL_BIO_SKIP_SEEDS))
       != len(CROSS_ATTENTION_3MODAL_BIO_SKIP_SEEDS)
):
    raise ValueError(
        "CROSS_ATTENTION_3MODAL_BIO_SKIP_SEEDS must be a non-empty list of unique integers."
    )

CROSS_ATTENTION_3MODAL_BIO_SKIP_CONTEXT = {
    "pipeline": pipeline,
    "TARGET_CONFIGS": CROSS_ATTENTION_3MODAL_BIO_SKIP_TARGET_CONFIGS,
    "optuna_objective_name": CROSS_ATTENTION_3MODAL_BIO_SKIP_OBJECTIVE_NAME,
    "datasets": DATASETS,
    "DNA_MODEL": DNA_MODEL,
    "PROTEIN_MODEL": PROTEIN_MODEL,
    "DEVICE": DEVICE,
    "token_embedding_cache_index": TOKEN_EMBEDDING_CACHE_INDEX,
    "token_batch_size": CROSS_ATTENTION_3MODAL_BIO_SKIP_BATCH_SIZE,
}
CROSS_ATTENTION_3MODAL_BIO_SKIP_RESULTS = {}
for _cross_skip_seed in CROSS_ATTENTION_3MODAL_BIO_SKIP_SEEDS:
    _cross_skip_config = dict(CROSS_ATTENTION_3MODAL_BIO_SKIP_CONFIG)
    _cross_skip_config["seed"] = int(_cross_skip_seed)
    _cross_skip_study_key = f"cross_attention_3modal_bio_skip_seed_{int(_cross_skip_seed)}"
    print(
        "Starting CrossAttention_ThreeModalities_BioSkip: "
        f"seed={_cross_skip_seed}; the direct bypass carries original bio-feature tokens only."
    )
    CROSS_ATTENTION_3MODAL_BIO_SKIP_RESULTS[int(_cross_skip_seed)] = (
        cross_attention_3modal_bio_skip_module.run_cross_attention_3modal_bio_skip_optuna(
            context=CROSS_ATTENTION_3MODAL_BIO_SKIP_CONTEXT,
            base_config=_cross_skip_config,
            search_space=CROSS_ATTENTION_3MODAL_BIO_SKIP_SEARCH_SPACE,
            n_trials=CROSS_ATTENTION_3MODAL_BIO_SKIP_N_TRIALS,
            display_fn=display,
            experiment_key=_cross_skip_study_key,
        )
    )


[I 2026-11-16 13:43:04,471] Using an existing study with `study_name='module06_cross_attention_3modal_bio_skip_seed_42'` instead of creating a new one.


Cross-Attention bio-skip target configs: ['B_with_geometry']
Starting CrossAttention_ThreeModalities_BioSkip: seed=42; the direct bypass carries original bio-feature tokens only.
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[CrossAttention_ThreeModalities_BioSkip] B_with_geometry epoch 1/30 train_loss=0.3109 val_loss=0.2119 val_MCC=0.7809 threshold=0.6233
[CrossAttention_ThreeModalities_BioSkip] B_with_geometry epoch 2/30 train_loss=0.2542 val_loss=0.1930 val_MCC=0.8039 threshold=0.5334
[CrossAttention_ThreeModalities_Bio

"[CrossAttention_ThreeModalities_BioSkip Optuna] optuna_0001; mean validation MCC=0.8077; A=nan; B=0.8077; params={'k_radius': 16, 'model_dim': 512, 'num_heads': 4, 'num_layers': 3, 'ffn_multiplier': 4, 'pooling': 'attention', 'dropout': 0.27731993941811406, 'attention_dropout': 0.09598658484197037, 'feature_dropout': 0.20312037280884873, 'learning_rate': 1.1141894188819903e-05, 'weight_decay': 3.4870787876478634e-05}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[RESUME] Reusing CrossAttention_ThreeModalities_BioSkip checkpoint for B_with_geometry
[CrossAttention_ThreeModalities_BioSkip] Reusing saved test metrics: B_with_geometry / Train3Val_Test
[CrossAttention_ThreeModalities_BioSkip] Reusing saved test metrics: B_with_geometry / Train3Val_ClinVarHQ
[CrossAttention_ThreeModalities_BioSkip] Reusing saved test metrics: B_with_geometry / Train3Val_UniProt
[CrossAttention_ThreeModalities_BioSkip] Reusing saved test metrics: B_with_geome

"[CrossAttention_ThreeModalities_BioSkip Optuna] optuna_0002; mean validation MCC=0.8077; A=nan; B=0.8077; params={'k_radius': 16, 'model_dim': 512, 'num_heads': 4, 'num_layers': 3, 'ffn_multiplier': 4, 'pooling': 'attention', 'dropout': 0.27731993941811406, 'attention_dropout': 0.09598658484197037, 'feature_dropout': 0.20312037280884873, 'learning_rate': 1.1141894188819903e-05, 'weight_decay': 3.4870787876478634e-05}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[CrossAttention_ThreeModalities_BioSkip] B_with_geometry epoch 1/30 train_loss=0.3096 val_loss=0.1853 val_MCC=0.7940 threshold=0.7089


KeyboardInterrupt: 

In [2]:
# Cross-Attention 3 Modality with Tokenwise Ref-Alt Difference.
# DNA/protein inputs are aligned alt-minus-ref token embeddings; bio/geometry is the third modality.
import importlib

for _token_diff_module_name in (
    "core.module05v2_fusion_classifier.evaluator_profiler",
    "core.module06_architecture_experiments.architecture_experiment_runner",
    "core.module06_architecture_experiments.token_topk_experiment",
    "core.module06_architecture_experiments.transformer_3modal_experiment",
    "core.module06_architecture_experiments.cross_attention_3modal_experiment",
    "core.module06_architecture_experiments.cross_attention_3modal_token_diff_experiment",
):
    importlib.reload(importlib.import_module(_token_diff_module_name))

cross_attention_3modal_token_diff_module = importlib.import_module(
    "core.module06_architecture_experiments.cross_attention_3modal_token_diff_experiment"
)
if "TOKEN_EMBEDDING_CACHE_INDEX" not in globals():
    token_cache_module = importlib.import_module(
        "core.module06_architecture_experiments.token_embedding_cache"
    )
    TOKEN_EMBEDDING_CACHE_INDEX = token_cache_module.load_token_embedding_cache_index(
        pipeline.batch_run_dir
    )

CROSS_ATTENTION_3MODAL_TOKEN_DIFF_CONFIG = dict(
    cross_attention_3modal_token_diff_module.DEFAULT_CROSS_ATTENTION_3MODAL_TOKEN_DIFF_CONFIG
)
CROSS_ATTENTION_3MODAL_TOKEN_DIFF_SEARCH_SPACE = dict(
    cross_attention_3modal_token_diff_module.DEFAULT_CROSS_ATTENTION_3MODAL_TOKEN_DIFF_SEARCH_SPACE
)
# Tune the variant-centered window, attention shape, regularization, and optimizer here.
CROSS_ATTENTION_3MODAL_TOKEN_DIFF_SEARCH_SPACE["k_radius"] = {
    "type": "categorical", "choices": [16]
}
CROSS_ATTENTION_3MODAL_TOKEN_DIFF_N_TRIALS = 1
CROSS_ATTENTION_3MODAL_TOKEN_DIFF_SEEDS = [42]
CROSS_ATTENTION_3MODAL_TOKEN_DIFF_BATCH_SIZE = 8
CROSS_ATTENTION_3MODAL_TOKEN_DIFF_RUN_CONFIG_A = False  # Set True to include A_no_geometry.
CROSS_ATTENTION_3MODAL_TOKEN_DIFF_TARGET_CONFIGS = [
    config for config in TARGET_CONFIGS
    if CROSS_ATTENTION_3MODAL_TOKEN_DIFF_RUN_CONFIG_A
    or config.get("name") != "A_no_geometry"
]
if not CROSS_ATTENTION_3MODAL_TOKEN_DIFF_TARGET_CONFIGS:
    raise ValueError("No target configurations selected for the token-difference study.")
print(
    "Cross-Attention token-difference target configs:",
    [config["name"] for config in CROSS_ATTENTION_3MODAL_TOKEN_DIFF_TARGET_CONFIGS],
)
_token_diff_config_names = {
    config["name"] for config in CROSS_ATTENTION_3MODAL_TOKEN_DIFF_TARGET_CONFIGS
}
if _token_diff_config_names == {"A_no_geometry", "B_with_geometry"}:
    CROSS_ATTENTION_3MODAL_TOKEN_DIFF_OBJECTIVE_NAME = "mean_validation_mcc_across_A_and_B"
elif len(_token_diff_config_names) == 1:
    CROSS_ATTENTION_3MODAL_TOKEN_DIFF_OBJECTIVE_NAME = (
        f"validation_mcc_on_{next(iter(_token_diff_config_names))}"
    )
else:
    CROSS_ATTENTION_3MODAL_TOKEN_DIFF_OBJECTIVE_NAME = (
        "mean_validation_mcc_over_selected_configs"
    )
if (
    not CROSS_ATTENTION_3MODAL_TOKEN_DIFF_SEEDS
    or len(set(CROSS_ATTENTION_3MODAL_TOKEN_DIFF_SEEDS))
       != len(CROSS_ATTENTION_3MODAL_TOKEN_DIFF_SEEDS)
):
    raise ValueError(
        "CROSS_ATTENTION_3MODAL_TOKEN_DIFF_SEEDS must be a non-empty list of unique integers."
    )

CROSS_ATTENTION_3MODAL_TOKEN_DIFF_CONTEXT = {
    "pipeline": pipeline,
    "TARGET_CONFIGS": CROSS_ATTENTION_3MODAL_TOKEN_DIFF_TARGET_CONFIGS,
    "optuna_objective_name": CROSS_ATTENTION_3MODAL_TOKEN_DIFF_OBJECTIVE_NAME,
    "datasets": DATASETS,
    "DNA_MODEL": DNA_MODEL,
    "PROTEIN_MODEL": PROTEIN_MODEL,
    "DEVICE": DEVICE,
    "token_embedding_cache_index": TOKEN_EMBEDDING_CACHE_INDEX,
    "token_batch_size": CROSS_ATTENTION_3MODAL_TOKEN_DIFF_BATCH_SIZE,
}
CROSS_ATTENTION_3MODAL_TOKEN_DIFF_RESULTS = {}
for _token_diff_seed in CROSS_ATTENTION_3MODAL_TOKEN_DIFF_SEEDS:
    _token_diff_config = dict(CROSS_ATTENTION_3MODAL_TOKEN_DIFF_CONFIG)
    _token_diff_config["seed"] = int(_token_diff_seed)
    _token_diff_study_key = f"cross_attention_3modal_token_diff_seed_{int(_token_diff_seed)}"
    print(
        "Starting CrossAttention_ThreeModalities_TokenDiff: "
        f"seed={_token_diff_seed}; each DNA/protein window token is alt minus ref "
        "at the matching variant-relative offset."
    )
    CROSS_ATTENTION_3MODAL_TOKEN_DIFF_RESULTS[int(_token_diff_seed)] = (
        cross_attention_3modal_token_diff_module.run_cross_attention_3modal_token_diff_optuna(
            context=CROSS_ATTENTION_3MODAL_TOKEN_DIFF_CONTEXT,
            base_config=_token_diff_config,
            search_space=CROSS_ATTENTION_3MODAL_TOKEN_DIFF_SEARCH_SPACE,
            n_trials=CROSS_ATTENTION_3MODAL_TOKEN_DIFF_N_TRIALS,
            display_fn=display,
            experiment_key=_token_diff_study_key,
        )
    )

Cross-Attention token-difference target configs: ['B_with_geometry']
Starting CrossAttention_ThreeModalities_TokenDiff: seed=42; each DNA/protein window token is alt minus ref at the matching variant-relative offset.


[I 2026-11-16 16:10:00,496] A new study created in RDB with name: module06_cross_attention_3modal_token_diff_seed_42


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[CrossAttention_ThreeModalities_TokenDiff] B_with_geometry epoch 1/30 train_loss=0.3554 val_loss=0.2029 val_MCC=0.7579 threshold=0.6612
[CrossAttention_ThreeModalities_TokenDiff] B_with_geometry epoch 2/30 train_loss=0.2956 val_loss=0.1975 val_MCC=0.7917 threshold=0.6415
[CrossAttention_ThreeModalities_TokenDiff] B_with_geometry epoch 3/30 train_loss=0.2714 val_loss=0.2649 val_MCC=0.7908 threshold=0.8847
[CrossAttention_ThreeModalities_TokenDiff] B_with_geometry epoch 4/30 trai

"[CrossAttention_ThreeModalities_TokenDiff Optuna] optuna_0000; mean validation MCC=0.8170; A=nan; B=0.8170; params={'k_radius': 16, 'model_dim': 512, 'num_heads': 4, 'num_layers': 3, 'ffn_multiplier': 4, 'pooling': 'mean', 'dropout': 0.27, 'attention_dropout': 0.0924, 'feature_dropout': 0.215, 'learning_rate': 1.1e-05, 'weight_decay': 0.000398}"

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Pooling,DNA_Model,Prot_Model,Network,Architecture,...,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_AUROC,Val_AUPRC,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
0,module06_architecture_run_20261104_071811,DL,validation,B_with_geometry,bio_core_dna_prot_spliceai_geom,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,CrossAttention_ThreeModalities_TokenDiff,CrossAttention_ThreeModalities_TokenDiff,...,0.8629,0.8401,0.9685,0.8513,0.9726,0.9099,optuna_0000,0,"{""attention_dropout"": 0.0924, ""dropout"": 0.27,...",0.817


,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Classifier_Peak_VRAM_MB,E2E_GFLOPs/Sample,E2E_Latency_ms/Sample,E2E_Total_Params,E2E_Param_Mem_MB,Max_Peak_VRAM_MB,Trial_ID,Optuna_Trial,Optuna_Hyperparameters,Optuna_Objective_Mean_Val_MCC
1,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,CrossAttention_ThreeModalities_TokenDiff,...,558.31,820.7994,39.06,1168705700,2290.1,1790.91,optuna_0000,0,"{""attention_dropout"": 0.0924, ""dropout"": 0.27,...",0.817
3,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_ProteinGym,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,CrossAttention_ThreeModalities_TokenDiff,...,558.31,820.7994,38.78,1168705700,2290.1,1790.91,optuna_0000,0,"{""attention_dropout"": 0.0924, ""dropout"": 0.27,...",0.817
0,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_Test,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,CrossAttention_ThreeModalities_TokenDiff,...,558.31,820.7994,38.60,1168705700,2290.1,1790.91,optuna_0000,0,"{""attention_dropout"": 0.0924, ""dropout"": 0.27,...",0.817
2,module06_architecture_run_20261104_071811,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,CrossAttention_ThreeModalities_TokenDiff,...,558.31,820.7994,38.89,1168705700,2290.1,1790.91,optuna_0000,0,"{""attention_dropout"": 0.0924, ""dropout"": 0.27,...",0.817


'CrossAttention_ThreeModalities_TokenDiff Optuna study complete. SQLite and CSV results: D:\\Missense_variant_pathogenicity_prediction\\experiments\\module06_architecture_run_20261104_071811\\module06_architecture_experiments\\cross_attention_3modal_token_diff_seed_42'

In [ ]:
# Dynamic Module 6 result reader: discovers every completed architecture study in this batch.
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Resolve the batch folder from several anchors. In some notebooks REPO_ROOT points
# to ``controllers``; therefore also search repository ancestors for the real batch.
_raw_batch_dir = Path(pipeline.batch_run_dir)
_module06_batch_name = _raw_batch_dir.name
_module06_anchors = [Path.cwd()]
for _anchor_value in (globals().get("REPO_ROOT"), globals().get("PROJECT_ROOT")):
    if _anchor_value:
        try:
            _module06_anchors.append(Path(_anchor_value))
        except TypeError:
            pass
if _raw_batch_dir.is_absolute():
    _module06_anchors.extend(_raw_batch_dir.parents)
_module06_anchors = [anchor.resolve() for anchor in _module06_anchors]

# Detect the checkout root by its repository folders, independently of notebook cwd.
_module06_repo_roots = []
for _anchor in _module06_anchors:
    for _candidate_root in (_anchor, *_anchor.parents):
        if ((_candidate_root / "core").is_dir()
                and (_candidate_root / "controllers").is_dir()):
            _module06_repo_roots.append(_candidate_root)
_module06_repo_roots = list(dict.fromkeys(_module06_repo_roots))

_module06_batch_candidates = []
if "RESUME_BATCH_RUN_DIR" in globals() and RESUME_BATCH_RUN_DIR:
    _resume_batch_path = Path(RESUME_BATCH_RUN_DIR)
    if _resume_batch_path.is_absolute():
        _module06_batch_candidates.append(_resume_batch_path)
    else:
        _module06_batch_candidates.extend(
            root / _resume_batch_path for root in _module06_repo_roots
        )
        _module06_batch_candidates.append(Path.cwd() / _resume_batch_path)
if _raw_batch_dir.is_absolute():
    _module06_batch_candidates.append(_raw_batch_dir)
else:
    _module06_batch_candidates.extend(
        root / _raw_batch_dir for root in _module06_repo_roots
    )
    _module06_batch_candidates.append(Path.cwd() / _raw_batch_dir)
_module06_batch_candidates.extend(
    root / "experiments" / _module06_batch_name for root in _module06_repo_roots
)
_module06_batch_candidates = list(dict.fromkeys(
    path.resolve() for path in _module06_batch_candidates
))
MODULE06_BATCH_DIR = next(
    (path for path in _module06_batch_candidates
     if (path / "module06_architecture_experiments").is_dir()),
    _module06_batch_candidates[0],
)
MODULE06_RESULT_ROOT = MODULE06_BATCH_DIR / "module06_architecture_experiments"
MODULE06_TOP_N_PER_DATASET = 10


def _module06_top_per_dataset(frame, limit):
    if "Dataset" not in frame.columns:
        return frame.sort_values("MCC", ascending=False, na_position="last").head(limit)
    return (
        frame.sort_values(
            ["Dataset", "MCC"], ascending=[True, False], na_position="last"
        )
        .groupby("Dataset", dropna=False, sort=False)
        .head(limit)
        .reset_index(drop=True)
    )


def _module06_candidate_id(frame):
    for column in ("Trial_ID", "Optuna_Trial", "Trial", "Candidate_ID", "Run_ID"):
        if column in frame.columns:
            values = frame[column].astype("string")
            if column == "Optuna_Trial":
                values = "optuna_" + values.fillna("single")
            return values.fillna("single")
    return pd.Series("single", index=frame.index, dtype="string")


def _module06_display_architecture(frame):
    if "Architecture" not in frame.columns:
        if "Network" in frame.columns:
            return frame["Network"].astype("string")
        return pd.Series("Unknown", index=frame.index, dtype="string")
    architecture = frame["Architecture"].astype("string").fillna("Unknown")
    variant = (
        frame["Variant"].astype("string").fillna("")
        if "Variant" in frame.columns
        else pd.Series("", index=frame.index, dtype="string")
    )
    # Older saved rows used a shared Architecture label; infer scope from Variant.
    mask = architecture.eq("CrossAttention_ThreeModalities")
    architecture.loc[mask] = "CrossAttention_ThreeModalities_VariantWindow"
    mask = architecture.eq("CrossAttention_ThreeModalities_InputScope")
    center = mask & variant.str.startswith("center_token")
    full = mask & variant.str.startswith("full_tokens_unpooled")
    architecture.loc[center] = "CrossAttention_ThreeModalities_CenterToken"
    architecture.loc[full] = "CrossAttention_ThreeModalities_FullTokens"
    return architecture


def _module06_model_name(frame, fallback):
    name = _module06_display_architecture(frame)
    return name.fillna(fallback).replace("", fallback)


def _module06_validation_path(test_path):
    name = test_path.name
    for suffix in ("_optuna_test_metrics.csv", "_grid_test_metrics.csv", "_test_metrics.csv"):
        if name.endswith(suffix):
            validation_suffix = suffix.replace("test_metrics", "validation_metrics")
            return test_path.with_name(name[:-len(suffix)] + validation_suffix)
    return None


MODULE06_TEST_FILES = []
if MODULE06_RESULT_ROOT.exists():
    for path in MODULE06_RESULT_ROOT.rglob("*_test_metrics.csv"):
        if "trials" in path.relative_to(MODULE06_RESULT_ROOT).parts:
            continue
        if path.name.endswith(("_optuna_test_metrics.csv", "_grid_test_metrics.csv")) or path.name.endswith("_test_metrics.csv"):
            MODULE06_TEST_FILES.append(path)
MODULE06_TEST_FILES = sorted(set(MODULE06_TEST_FILES))

_module06_test_frames = []
_module06_validation_frames = []
_module06_read_errors = []
for _test_path in MODULE06_TEST_FILES:
    _relative_key = str(_test_path.relative_to(MODULE06_RESULT_ROOT))
    _search_method = (
        "Optuna" if "_optuna_test_metrics.csv" in _test_path.name
        else "Grid" if "_grid_test_metrics.csv" in _test_path.name
        else "Single run"
    )
    try:
        _test = pd.read_csv(_test_path)
    except Exception as _error:
        _module06_read_errors.append(f"{_test_path}: {_error}")
        continue
    if _test.empty or "MCC" not in _test.columns:
        continue
    _test["MCC"] = pd.to_numeric(_test["MCC"], errors="coerce")
    _test = _test.loc[_test["MCC"].notna()].copy()
    if _test.empty:
        continue
    _test["_Source_Key"] = _relative_key
    _test["Architecture"] = _module06_display_architecture(_test)
    _test["_Candidate_ID"] = _module06_candidate_id(_test)
    _test["_Model_Name"] = _module06_model_name(_test, _test_path.parent.name)
    _test["_Study"] = _test_path.parent.name
    _test["Search_Method"] = _search_method
    _test["Result_File"] = _relative_key
    _module06_test_frames.append(_test)

    _validation_path = _module06_validation_path(_test_path)
    if _validation_path is None or not _validation_path.exists():
        continue
    try:
        _validation = pd.read_csv(_validation_path)
    except Exception as _error:
        _module06_read_errors.append(f"{_validation_path}: {_error}")
        continue
    if _validation.empty or "Val_MCC" not in _validation.columns:
        continue
    _validation["Val_MCC"] = pd.to_numeric(_validation["Val_MCC"], errors="coerce")
    _validation = _validation.loc[_validation["Val_MCC"].notna()].copy()
    if _validation.empty:
        continue
    _validation["_Source_Key"] = _relative_key
    _validation["Architecture"] = _module06_display_architecture(_validation)
    _validation["_Candidate_ID"] = _module06_candidate_id(_validation)
    _validation["_Model_Name"] = _module06_model_name(_validation, _test_path.parent.name)
    _module06_validation_frames.append(_validation)

if not _module06_test_frames:
    MODULE06_ARCHITECTURE_RESULTS_ALL = pd.DataFrame()
    MODULE06_ARCHITECTURE_RESULTS_SELECTED = pd.DataFrame()
    print(f"No completed Module 6 test metrics found in: {MODULE06_RESULT_ROOT}")
else:
    MODULE06_ARCHITECTURE_RESULTS_ALL = pd.concat(_module06_test_frames, ignore_index=True, sort=False)
    MODULE06_ARCHITECTURE_RESULTS_ALL = MODULE06_ARCHITECTURE_RESULTS_ALL.sort_values(
        "MCC", ascending=False, na_position="last"
    ).reset_index(drop=True)

    if _module06_validation_frames:
        _validation_all = pd.concat(_module06_validation_frames, ignore_index=True, sort=False)
        _validation_keys = ["_Source_Key", "_Model_Name", "_Candidate_ID"]
        if "Target_Config" not in _validation_all.columns:
            _validation_all["Target_Config"] = "unknown"
        _validation_all = _validation_all.drop_duplicates(
            subset=_validation_keys + ["Target_Config"], keep="last"
        )
        _candidate_scores = (
            _validation_all.groupby(_validation_keys, dropna=False)["Val_MCC"]
            .mean().rename("Mean_Val_MCC_A_B").reset_index()
        )
        _best_candidate_keys = (
            _candidate_scores.sort_values("Mean_Val_MCC_A_B", ascending=False)
            .drop_duplicates(subset=["_Source_Key", "_Model_Name"], keep="first")
        )
        MODULE06_ARCHITECTURE_RESULTS_SELECTED = MODULE06_ARCHITECTURE_RESULTS_ALL.merge(
            _best_candidate_keys,
            on=["_Source_Key", "_Model_Name", "_Candidate_ID"],
            how="inner",
        )
        MODULE06_ARCHITECTURE_RESULTS_SELECTED = MODULE06_ARCHITECTURE_RESULTS_SELECTED.sort_values(
            "MCC", ascending=False, na_position="last"
        ).reset_index(drop=True)
    else:
        MODULE06_ARCHITECTURE_RESULTS_SELECTED = pd.DataFrame()

    _preferred_columns = [
        "Dataset", "Target_Config", "_Model_Name", "Architecture", "Variant",
        "Search_Method", "Trial_ID", "Optuna_Trial", "Mean_Val_MCC_A_B", "Input_Representation",
        "Decision_Threshold", "MCC", "AUROC", "AUPRC", "Accuracy", "F1_Score",
        "Optuna_Hyperparameters", "Grid_Hyperparameters", "Result_File",
    ]
    _selected_display_columns = [
        column for column in _preferred_columns
        if column in MODULE06_ARCHITECTURE_RESULTS_SELECTED.columns
    ]
    print(
        f"Discovered {len(MODULE06_TEST_FILES)} result files; "
        f"loaded {len(_module06_test_frames)} completed study result sets; "
        f"{MODULE06_ARCHITECTURE_RESULTS_ALL['_Model_Name'].nunique()} model names."
    )
    if not MODULE06_ARCHITECTURE_RESULTS_SELECTED.empty:
        print("Best validation-selected trial per architecture/study; rows sorted by test MCC descending:")
        with pd.option_context("display.max_rows", None, "display.max_columns", None):
            display(MODULE06_ARCHITECTURE_RESULTS_SELECTED[_selected_display_columns])
    else:
        print("Test results were read, but no matching validation CSV was available to select trials.")
        display(_module06_top_per_dataset(MODULE06_ARCHITECTURE_RESULTS_ALL, MODULE06_TOP_N_PER_DATASET))

    print(
        f"Top {MODULE06_TOP_N_PER_DATASET} completed trial/test rows per test dataset "
        "across all discovered studies, sorted by test MCC:"
    )
    _all_display_columns = [
        column for column in _preferred_columns
        if column in MODULE06_ARCHITECTURE_RESULTS_ALL.columns
    ]
    _top_rows_per_dataset = _module06_top_per_dataset(
        MODULE06_ARCHITECTURE_RESULTS_ALL, MODULE06_TOP_N_PER_DATASET
    )
    display(_top_rows_per_dataset[_all_display_columns])

    if not MODULE06_ARCHITECTURE_RESULTS_SELECTED.empty:
        _plot_data = MODULE06_ARCHITECTURE_RESULTS_SELECTED.copy()
        if "Dataset" in _plot_data.columns and "Target_Config" in _plot_data.columns:
            _plot_data["Plot_Model"] = (
                _plot_data["_Model_Name"].astype(str)
                + " [" + _plot_data["Search_Method"].astype(str) + "]"
                + " / " + _plot_data["Target_Config"].astype(str)
            )
            _datasets_to_plot = list(pd.unique(_plot_data["Dataset"].dropna()))
            if _datasets_to_plot:
                _ncols = 2
                _nrows = math.ceil(len(_datasets_to_plot) / _ncols)
                _nlabels = max(
                    int((_plot_data["Dataset"] == _dataset).sum())
                    for _dataset in _datasets_to_plot
                )
                _fig, _axes = plt.subplots(
                    _nrows, _ncols,
                    figsize=(18, max(7, 0.34 * _nlabels + 2)),
                    squeeze=False,
                )
                _config_colors = {
                    "A_no_geometry": "#3572A5",
                    "B_with_geometry": "#E07A2D",
                }
                for _index, _dataset in enumerate(_datasets_to_plot):
                    _ax = _axes.flat[_index]
                    _part = _plot_data.loc[_plot_data["Dataset"] == _dataset].copy()
                    _part = _part.sort_values("MCC", ascending=True)
                    _colors = [
                        _config_colors.get(str(config), "#6C757D")
                        for config in _part["Target_Config"]
                    ]
                    _bars = _ax.barh(_part["Plot_Model"], _part["MCC"], color=_colors)
                    _ax.set_title(str(_dataset))
                    _ax.set_xlabel("MCC (test)")
                    _ax.set_ylabel("")
                    _ax.grid(axis="x", alpha=0.25)
                    _ax.tick_params(axis="y", labelsize=8)
                    _ax.set_xlim(min(-0.1, float(_part["MCC"].min()) - 0.04),
                                 max(0.1, float(_part["MCC"].max()) + 0.08))
                    for _bar, _value in zip(_bars, _part["MCC"]):
                        _ax.text(_value, _bar.get_y() + _bar.get_height() / 2,
                                 f" {_value:.3f}", va="center", fontsize=7)
                    _ax.invert_yaxis()
                for _index in range(len(_datasets_to_plot), _nrows * _ncols):
                    _axes.flat[_index].set_visible(False)
                _legend = [
                    Patch(facecolor=color, label=label)
                    for label, color in _config_colors.items()
                ]
                _fig.legend(handles=_legend, loc="upper center", ncol=2, frameon=False)
                _fig.suptitle("Module 6: validation-selected architecture trials on test sets", y=1.01)
                _fig.tight_layout()
                plt.show()

if _module06_read_errors:
    print("Some result files could not be read:")
    for _message in _module06_read_errors:
        print(" -", _message)
